ETO NA TLGA GUYS FINAL NA

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1MJByxP2BQyGAQTFvGxPMg8vTuIto2FuN

# TIBOK — Combined MIT-BIH + INCART, RR-Interval-Fused 1D-CNN (Colab-ready)

Trains on the **combined MIT-BIH + St. Petersburg INCART Arrhythmia Databases**, patient-level
70% train / 15% val / 15% test split — exactly the protocol already described in Section C
("Model Architecture Training") and Section XI ("Data Collection") of the pre-oral document.
INCART's 75 twelve-lead recordings are resampled from their native 257 Hz to the shared 360 Hz
working rate, with lead II pulled as the MLII-equivalent channel per the document's stated plan.

Architecturally this is identical to `TIBOK_RR_CNN.ipynb` (the MIT-BIH-only ablation kept
alongside this notebook for comparison): each ECG window is paired with 4 RR-interval features
(pre-RR, post-RR, causal local-average RR, prematurity ratio), fused into the classifier head
next to the CNN's pooled output. The only change here is the training population — MIT-BIH
alone left the network under-exposed to how "normal" and "abnormal" morphology look across
different hospitals/equipment, which capped precision on held-out patients. Pooling in INCART's
more varied population is the standard fix for that inter-patient generalization gap (Qi et al.,
2023, cited in the RRL, report the same effect from pooling these two databases).

**Architecture**: ECG window (1250×1) → 3×Conv1D(16/32/64, k7, BN+ReLU+SpatialDropout) →
GlobalAveragePooling1D(64) ⊕ RR features (4) → Dense(16) → Concat(80) → Dense(32) → Dropout →
Output(1, sigmoid).

In [ ]:
!pip install -q wfdb

from google.colab import drive
drive.mount('/content/drive')  # required below — MIT-BIH/INCART are cached here, not streamed

import os, json, time
import numpy as np
import pandas as pd
import wfdb
import tensorflow as tf
from tensorflow.keras import layers, models, Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)
tf.random.set_seed(42)
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
tf.config.experimental.enable_op_determinism()

RUN_TAG = "tibok_combined_rr_cnn"
print(f"RUN_TAG = {RUN_TAG}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")
if not gpus:
    print("No GPU detected — Runtime > Change runtime type > GPU will speed this up substantially.")

## Write the `tibok` modules onto the runtime

This notebook is self-contained on purpose. The repository is private, so a
`git clone` from a Colab runtime would prompt for credentials and fail, and
`files.upload()` would mean re-uploading the modules every time the runtime is
recycled. The cells below write them straight to disk instead.

- `data_paths.py` — resolves the MIT-BIH / INCART folders and fails loudly on a
  bad path. Used by the data-loading section immediately below.
- `quantization.py` — INT8 conversion and FP32-vs-INT8 verification, used at the
  end of the notebook.
- `thresholds.py` / `trials.py` — operating-point selection and the optional
  multi-trial replication study.

**Do not edit the modules here.** They are generated from `tibok/` in the repo by
`tools/build_notebook.py`; edits made in these cells are lost the next time the
notebook is rebuilt. Change the repo file and re-run the builder.

In [ ]:
import os, sys
os.makedirs('tibok', exist_ok=True)
# Lazy __init__: importing tibok.data_paths must not drag in TensorFlow.
open('tibok/__init__.py', 'w').close()
if '.' not in sys.path:
    sys.path.insert(0, '.')

In [ ]:
%%writefile tibok/quantization.py
"""
TIBOK — INT8 post-training quantization + verification for the RR-fused 1D-CNN.

Companion to `eto_na_tlga_guys_final_na.py`. That notebook trains the model; this
module takes the trained Keras model plus the held-out arrays and answers the
deployment questions the pre-oral commits to:

  RQ2.1 / RQ2.3  Does INT8 quantization change F1 / specificity on held-out patients?
                 -> `compare_fp32_int8` runs a *paired* comparison (McNemar's exact
                    test + bootstrap CIs on the deltas), not two unrelated metric
                    dumps, because both models score the identical test beats.

  Fit on nRF52840 (1 MB flash / 256 KB RAM)
                 -> `estimate_tflm_arena` walks the operator schedule and reports the
                    peak simultaneously-live activation set, which is what the
                    TFLite-Micro arena actually has to hold.

Why this exists as its own module rather than one more notebook cell: the original
cell silently fell back to dynamic-range (weight-only) quantization when full-integer
conversion failed. That fallback is not deployable here -- TFLite-Micro has no
dynamic-range kernels for Conv1D/FullyConnected, so a dynamic-range .tflite will fail
at `AllocateTensors()` on the nRF52840 even though it loads fine on desktop. Falling
back quietly turns a conversion bug into a firmware bug found weeks later. This module
tries several full-integer conversion strategies in order and only ever reports
dynamic-range as an explicit, loudly-flagged failure state.
"""

from __future__ import annotations

import json
import os
import tempfile
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)

__all__ = [
    "QuantizationResult", "quantize_int8", "evaluate_at_threshold", "recalibrate_int8",
    "compare_fp32_int8", "estimate_tflm_arena", "run_tflite",
    "benchmark_latency", "export_c_header", "quantize_and_test",
]

NRF52840_FLASH_KB = 1024
NRF52840_RAM_KB = 256


class _quiet:
    """Silence the SavedModel-export chatter the converter prints on every call.

    TFLite conversion internally exports a SavedModel and dumps its full signature and
    every captured resource tensor to stdout. With three conversion strategies plus a
    probe that is several hundred lines of noise per run, which buries the numbers this
    module exists to report. Redirect at the file-descriptor level, since the output
    comes from C++ rather than Python's `sys.stdout`.
    """

    def __enter__(self):
        import sys
        sys.stdout.flush()
        self._saved = os.dup(1)
        self._null = os.open(os.devnull, os.O_WRONLY)
        os.dup2(self._null, 1)
        return self

    def __exit__(self, *exc):
        import sys
        sys.stdout.flush()
        os.dup2(self._saved, 1)
        os.close(self._null)
        os.close(self._saved)
        return False


# ---------------------------------------------------------------------------
# Input identification
# ---------------------------------------------------------------------------

def _classify_inputs(input_details, window_size, n_rr_features=4):
    """Return (ecg_index, rr_index) into `input_details`.

    The original notebook guessed with `shape[-1] != 4`, which silently breaks if the
    ECG window ever ends in a 4-sized axis. Match on name first (the Keras layers are
    named `ecg_window` / `rr_features`), then fall back to rank/shape, which is
    unambiguous here: the ECG tensor is rank 3 (N, window, 1), RR is rank 2 (N, 4).
    """
    ecg = rr = None
    for i, d in enumerate(input_details):
        name = d["name"].lower()
        if "ecg" in name or "window" in name:
            ecg = i
        elif "rr" in name:
            rr = i
    if ecg is not None and rr is not None and ecg != rr:
        return ecg, rr

    ecg = rr = None
    for i, d in enumerate(input_details):
        shape = [int(s) for s in d["shape"]]
        if len(shape) == 3 and window_size in shape:
            ecg = i
        elif len(shape) == 2 and shape[-1] == n_rr_features:
            rr = i
    if ecg is None or rr is None or ecg == rr:
        raise RuntimeError(
            f"Could not identify ECG vs RR inputs from {[(d['name'], d['shape']) for d in input_details]}"
        )
    return ecg, rr


# ---------------------------------------------------------------------------
# Conversion
# ---------------------------------------------------------------------------

@dataclass
class QuantizationResult:
    tflite_bytes: bytes = b""
    full_int8: bool = False
    strategy: str = ""
    attempts: list = field(default_factory=list)
    size_kb: float = 0.0
    ecg_scale: float = None
    ecg_zero_point: int = None
    rr_scale: float = None
    rr_zero_point: int = None
    output_scale: float = None
    output_zero_point: int = None

    def summary(self):
        d = asdict(self)
        d.pop("tflite_bytes")
        return d


def _representative_factory(X_val, RR_val_n, order, n_samples=800, seed=42):
    """Yield calibration samples in the order the *converter* expects them.

    This is the fix for the `input->dims->size != 4 (3 != 4)` calibrator crash. With
    Keras 3 (TF >= 2.16) the concrete function traced out of a multi-input functional
    model does not necessarily order its flat input list the same way `model.inputs`
    does. When the order is flipped, the calibrator pushes the rank-2 RR tensor into
    the Conv1D branch, whose kernel expects rank 4 after the implicit ExpandDims --
    hence `3 != 4`. An untrained model converts fine only because the failure needs a
    calibration pass to happen at all, which is why the bug looked weight-dependent.

    `order` is derived by inspecting a throwaway float conversion of the same graph,
    so we feed the tensors in whatever order that specific TF build actually produced.
    """
    rng = np.random.default_rng(seed)
    n = min(n_samples, len(X_val))
    idx = rng.choice(len(X_val), n, replace=False)

    def representative_dataset():
        for i in idx:
            sample = {
                "ecg": X_val[i:i + 1].astype(np.float32),
                "rr": RR_val_n[i:i + 1].astype(np.float32),
            }
            yield [sample[k] for k in order]

    return representative_dataset


def _input_order_from_float_model(model, window_size):
    """Convert once without quantization to learn this TF build's input ordering."""
    with _quiet():
        conv = tf.lite.TFLiteConverter.from_keras_model(model)
        float_model = conv.convert()
    interp = tf.lite.Interpreter(model_content=float_model)
    interp.allocate_tensors()
    details = interp.get_input_details()
    ecg_i, rr_i = _classify_inputs(details, window_size)
    order = [None, None]
    order[ecg_i] = "ecg"
    order[rr_i] = "rr"
    return order, len(float_model)


def _apply_int8_settings(converter, rep_ds):
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = rep_ds
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    return converter


def quantize_int8(model, X_val, RR_val_n, window_size, n_calib=800, seed=42,
                  allow_dynamic_range_fallback=False):
    """Convert `model` to a full-integer INT8 .tflite, trying strategies in order.

    Set `allow_dynamic_range_fallback=True` only to unblock desktop experimentation.
    The resulting model will NOT run under TFLite-Micro on the nRF52840.
    """
    result = QuantizationResult()

    try:
        order, float_size = _input_order_from_float_model(model, window_size)
        result.attempts.append(f"probe: float conversion OK ({float_size/1024:.1f} KB), input order = {order}")
    except Exception as e:
        order = ["ecg", "rr"]
        result.attempts.append(f"probe: float conversion failed ({e!r}); assuming order {order}")

    rep_ds = _representative_factory(X_val, RR_val_n, order, n_calib, seed)

    # Strategy A -- direct from the Keras model, with calibration fed in probe order.
    def _strategy_keras():
        with _quiet():
            return _apply_int8_settings(tf.lite.TFLiteConverter.from_keras_model(model), rep_ds).convert()

    # Strategy B -- go through a SavedModel with an explicit batch-1 signature. This
    # pins the input spec instead of relying on whatever the Keras tracer emits, and
    # is the path TF documents for TF >= 2.16.
    def _strategy_savedmodel():
        with tempfile.TemporaryDirectory() as tmp, _quiet():
            path = os.path.join(tmp, "saved_model")
            if hasattr(model, "export"):
                model.export(path)
            else:
                tf.saved_model.save(model, path)
            conv = tf.lite.TFLiteConverter.from_saved_model(path)
            return _apply_int8_settings(conv, rep_ds).convert()

    # Strategy C -- build our own concrete function with a fixed, explicitly ordered
    # batch-1 signature. Batch 1 also matches how the firmware will actually invoke it.
    def _strategy_concrete():
        @tf.function(input_signature=[
            tf.TensorSpec([1, window_size, 1], tf.float32, name="ecg_window"),
            tf.TensorSpec([1, 4], tf.float32, name="rr_features"),
        ])
        def serve(ecg, rr):
            return model([ecg, rr], training=False)

        with _quiet():
            cf = serve.get_concrete_function()
            conv = tf.lite.TFLiteConverter.from_concrete_functions([cf], model)
            return _apply_int8_settings(conv, rep_ds).convert()

    strategies = [
        ("keras_direct", _strategy_keras),
        ("saved_model", _strategy_savedmodel),
        ("concrete_function_batch1", _strategy_concrete),
    ]

    for name, fn in strategies:
        try:
            blob = fn()
            result.tflite_bytes = blob
            result.full_int8 = True
            result.strategy = name
            result.attempts.append(f"{name}: SUCCESS (full integer)")
            break
        except Exception as e:
            result.attempts.append(f"{name}: failed -- {type(e).__name__}: {e}")

    if not result.full_int8:
        if not allow_dynamic_range_fallback:
            raise RuntimeError(
                "All full-integer quantization strategies failed:\n  - "
                + "\n  - ".join(result.attempts)
                + "\n\nNot falling back to dynamic-range: TFLite-Micro has no dynamic-range "
                  "kernels for this graph, so such a model cannot run on the nRF52840. "
                  "Pass allow_dynamic_range_fallback=True only for desktop experiments."
            )
        with _quiet():
            conv = tf.lite.TFLiteConverter.from_keras_model(model)
            conv.optimizations = [tf.lite.Optimize.DEFAULT]
            result.tflite_bytes = conv.convert()
        result.strategy = "dynamic_range_NOT_DEPLOYABLE"
        result.attempts.append("dynamic_range: used as explicit fallback -- NOT deployable on nRF52840")

    result.size_kb = len(result.tflite_bytes) / 1024

    interp = tf.lite.Interpreter(model_content=result.tflite_bytes)
    interp.allocate_tensors()
    in_det, out_det = interp.get_input_details(), interp.get_output_details()
    if result.full_int8:
        ecg_i, rr_i = _classify_inputs(in_det, window_size)
        result.ecg_scale, result.ecg_zero_point = (float(in_det[ecg_i]["quantization"][0]),
                                                   int(in_det[ecg_i]["quantization"][1]))
        result.rr_scale, result.rr_zero_point = (float(in_det[rr_i]["quantization"][0]),
                                                 int(in_det[rr_i]["quantization"][1]))
        result.output_scale, result.output_zero_point = (float(out_det[0]["quantization"][0]),
                                                         int(out_det[0]["quantization"][1]))
    return result


# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------

def run_tflite(qr: QuantizationResult, X, RR_n, window_size, batch_one=True):
    """Run the .tflite model over (X, RR_n) and return float probabilities.

    `batch_one=True` invokes one window at a time, which is exactly how the firmware
    will call it. Batched inference can differ slightly because it changes nothing
    numerically here but does exercise a different kernel path; keeping batch 1 means
    the numbers reported are the numbers the device will produce.
    """
    interp = tf.lite.Interpreter(model_content=qr.tflite_bytes)
    in_det = interp.get_input_details()
    out_det = interp.get_output_details()
    ecg_i, rr_i = _classify_inputs(in_det, window_size)

    n = len(X)
    if batch_one:
        interp.resize_tensor_input(in_det[ecg_i]["index"], [1, window_size, 1])
        interp.resize_tensor_input(in_det[rr_i]["index"], [1, RR_n.shape[1]])
    else:
        interp.resize_tensor_input(in_det[ecg_i]["index"], [n, window_size, 1])
        interp.resize_tensor_input(in_det[rr_i]["index"], [n, RR_n.shape[1]])
    interp.allocate_tensors()
    in_det, out_det = interp.get_input_details(), interp.get_output_details()

    def _prep(arr, det):
        if det["dtype"] == np.int8:
            s, z = det["quantization"]
            return np.clip(np.round(arr / s + z), -128, 127).astype(np.int8)
        return arr.astype(np.float32)

    Xq = _prep(X.reshape(n, window_size, 1), in_det[ecg_i])
    RRq = _prep(RR_n, in_det[rr_i])

    raw_codes = np.empty(n, dtype=np.float64)
    if batch_one:
        for i in range(n):
            interp.set_tensor(in_det[ecg_i]["index"], Xq[i:i + 1])
            interp.set_tensor(in_det[rr_i]["index"], RRq[i:i + 1])
            interp.invoke()
            raw_codes[i] = interp.get_tensor(out_det[0]["index"]).flatten()[0]
    else:
        interp.set_tensor(in_det[ecg_i]["index"], Xq)
        interp.set_tensor(in_det[rr_i]["index"], RRq)
        interp.invoke()
        raw_codes = interp.get_tensor(out_det[0]["index"]).flatten().astype(np.float64)

    if out_det[0]["dtype"] == np.int8:
        s, z = out_det[0]["quantization"]
        return (raw_codes - z) * s, raw_codes
    return raw_codes, raw_codes


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------

def evaluate_at_threshold(y_true, probs, threshold, label=None, verbose=True):
    pred = (probs > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    m = dict(
        threshold=float(threshold),
        accuracy=float(accuracy_score(y_true, pred)),
        precision=float(precision_score(y_true, pred, zero_division=0)),
        sensitivity=float(recall_score(y_true, pred, zero_division=0)),
        specificity=float(tn / (tn + fp)) if (tn + fp) else 0.0,
        f1_score=float(f1_score(y_true, pred, zero_division=0)),
        roc_auc=float(roc_auc_score(y_true, probs)),
        pr_auc=float(average_precision_score(y_true, probs)),
        tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
    )
    if verbose and label:
        print(f"\n--- {label} (threshold={threshold:.4f}) ---")
        for k, v in m.items():
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    return m


def _mcnemar_exact(b, c):
    """Two-sided exact McNemar p-value for discordant counts (b, c).

    b = FP32 correct & INT8 wrong, c = FP32 wrong & INT8 correct. Under H0 (quantization
    changes nothing systematic) each discordant beat is a fair coin flip, so the exact
    binomial tail is the right test -- and it is the *paired* test, which matters because
    both models are scored on the identical beats. Comparing two independent-sample
    confidence intervals here would be needlessly conservative.
    """
    from math import comb
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(comb(n, i) for i in range(k + 1)) / (2 ** n)
    return float(min(1.0, 2 * tail))


def _bootstrap_delta(y_true, probs_a, probs_b, threshold, metric, n_boot=2000, seed=42):
    """Percentile CI for (metric_b - metric_a), resampling beats in paired fashion."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    pred_a = (probs_a > threshold).astype(int)
    pred_b = (probs_b > threshold).astype(int)

    def _m(yt, pa):
        tn, fp, fn, tp = confusion_matrix(yt, pa, labels=[0, 1]).ravel()
        if metric == "f1":
            return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
        if metric == "specificity":
            return tn / (tn + fp) if (tn + fp) else 0.0
        if metric == "sensitivity":
            return tp / (tp + fn) if (tp + fn) else 0.0
        raise ValueError(metric)

    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        deltas[i] = _m(y_true[idx], pred_b[idx]) - _m(y_true[idx], pred_a[idx])
    return dict(
        observed=float(_m(y_true, pred_b) - _m(y_true, pred_a)),
        ci_lo=float(np.percentile(deltas, 2.5)),
        ci_hi=float(np.percentile(deltas, 97.5)),
    )


def compare_fp32_int8(y_true, probs_fp32, probs_int8, threshold, label="", n_boot=2000):
    """Paired FP32-vs-INT8 comparison at one operating point. Answers RQ2.1 / RQ2.3."""
    pred_a = (probs_fp32 > threshold).astype(int)
    pred_b = (probs_int8 > threshold).astype(int)
    correct_a, correct_b = (pred_a == y_true), (pred_b == y_true)
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    p = _mcnemar_exact(b, c)

    out = dict(
        label=label, threshold=float(threshold),
        fp32=evaluate_at_threshold(y_true, probs_fp32, threshold, verbose=False),
        int8=evaluate_at_threshold(y_true, probs_int8, threshold, verbose=False),
        mcnemar=dict(
            fp32_right_int8_wrong=b, fp32_wrong_int8_right=c,
            n_discordant=b + c, p_value=p,
            significant_at_0p05=bool(p < 0.05),
        ),
        label_agreement=float(np.mean(pred_a == pred_b)),
        prob_delta=dict(
            mean_abs=float(np.mean(np.abs(probs_int8 - probs_fp32))),
            max_abs=float(np.max(np.abs(probs_int8 - probs_fp32))),
        ),
        delta_f1=_bootstrap_delta(y_true, probs_fp32, probs_int8, threshold, "f1", n_boot),
        delta_specificity=_bootstrap_delta(y_true, probs_fp32, probs_int8, threshold, "specificity", n_boot),
        delta_sensitivity=_bootstrap_delta(y_true, probs_fp32, probs_int8, threshold, "sensitivity", n_boot),
    )
    return out


def print_comparison(cmp):
    f, q = cmp["fp32"], cmp["int8"]
    print(f"\n=== FP32 vs INT8 @ {cmp['label']} (threshold={cmp['threshold']:.4f}) ===")
    print(f"{'metric':<14}{'FP32':>10}{'INT8':>10}{'delta':>10}")
    for k in ("f1_score", "specificity", "sensitivity", "precision", "accuracy", "roc_auc", "pr_auc"):
        print(f"{k:<14}{f[k]:>10.4f}{q[k]:>10.4f}{q[k]-f[k]:>+10.4f}")
    m = cmp["mcnemar"]
    print(f"\nLabel agreement: {cmp['label_agreement']*100:.3f}%   "
          f"mean|dp|={cmp['prob_delta']['mean_abs']:.5f}  max|dp|={cmp['prob_delta']['max_abs']:.5f}")
    print(f"McNemar: FP32-right/INT8-wrong={m['fp32_right_int8_wrong']}, "
          f"FP32-wrong/INT8-right={m['fp32_wrong_int8_right']}, p={m['p_value']:.4f} "
          f"-> {'SIGNIFICANT difference' if m['significant_at_0p05'] else 'no significant difference'}")
    for name in ("delta_f1", "delta_specificity", "delta_sensitivity"):
        d = cmp[name]
        print(f"  {name:<18} {d['observed']:+.4f}  95% CI [{d['ci_lo']:+.4f}, {d['ci_hi']:+.4f}]")


def recalibrate_int8(qr, model, X_val, RR_val_n, y_val, window_size, batch_one=False,
                     precision_floor=0.90, target_sensitivity=0.90):
    """Re-select operating points on the INT8 model's OWN validation probabilities.

    The pipeline otherwise chooses thresholds from FP32 validation probabilities and then
    applies those same numbers to INT8 test scores. For answering "did quantization change
    the predictions" that is exactly right -- holding the threshold fixed is what makes it
    a controlled comparison.

    For deciding what to ship it is wrong, and pessimistically so. INT8 shifts the score
    distribution (here: systematically more positive), so an FP32-derived cutoff sits in
    the wrong place on the INT8 curve and throws away precision that the model has not
    actually lost. The giveaway is ROC-AUC: if it barely moves under quantization while F1
    drops sharply, the ranking survived and only the calibration moved -- and calibration
    is free to fix, because on-device you would tune the threshold against the INT8 model
    anyway.

    Returns thresholds chosen on INT8 validation scores, to be applied to INT8 test scores.
    Report both: the fixed-threshold comparison answers the research question, the
    recalibrated one describes the deployed device.
    """
    from .thresholds import select_thresholds
    val_probs_int8, _ = run_tflite(qr, X_val, RR_val_n, window_size, batch_one=batch_one)
    return select_thresholds(y_val, val_probs_int8,
                             precision_floor=precision_floor,
                             target_sensitivity=target_sensitivity)


# ---------------------------------------------------------------------------
# Memory footprint
# ---------------------------------------------------------------------------

_DTYPE_BYTES = {np.int8: 1, np.uint8: 1, np.int16: 2, np.uint16: 2,
                np.int32: 4, np.uint32: 4, np.int64: 8, np.float32: 4, np.float16: 2}


def _tensor_bytes(t):
    shape = t["shape"]
    n = int(np.prod(shape)) if getattr(shape, "size", len(shape)) > 0 else 1
    return n * _DTYPE_BYTES.get(t["dtype"], 4)


def estimate_tflm_arena(qr: QuantizationResult, window_size):
    """Estimate the TFLite-Micro tensor arena: peak *simultaneously live* activations.

    The original notebook summed every tensor in the model and called that RAM. That
    overstates the requirement twice over:

      1. It counts weight tensors, which TFLite-Micro reads straight out of the flash
         image (the model is a `const` array) and never copies into the arena.
      2. It ignores lifetime reuse. An activation is dead once its last consumer has
         run, and the arena planner reuses that space, so the requirement is the peak
         of the live set across the op schedule, not the total ever allocated.

    Walking the schedule instead gives a number in the right ballpark for sizing
    `kTensorArenaSize`. It is still an estimate -- the real planner adds per-tensor
    bookkeeping, 16-byte alignment padding, and scratch buffers some kernels request --
    so budget headroom and confirm against what `AllocateTensors()` actually reports on
    hardware before treating it as final.
    """
    interp = tf.lite.Interpreter(model_content=qr.tflite_bytes)
    interp.resize_tensor_input(interp.get_input_details()[0]["index"],
                               interp.get_input_details()[0]["shape"])
    interp.allocate_tensors()
    tensors = interp.get_tensor_details()
    by_index = {t["index"]: t for t in tensors}

    # Constant (weight) tensors live in flash. A tensor is constant if the interpreter
    # can hand us its contents without an invoke and no operator produces it.
    try:
        ops = interp._get_ops_details()
    except Exception as e:
        total = sum(_tensor_bytes(t) for t in tensors)
        return dict(method="fallback_sum_all_tensors", note=f"op schedule unavailable ({e!r})",
                    peak_arena_bytes=total, peak_arena_kb=total / 1024,
                    naive_sum_kb=total / 1024, flash_kb=qr.size_kb)

    produced_at, last_used_at = {}, {}
    for step, op in enumerate(ops):
        for ti in op["inputs"]:
            if ti >= 0:
                last_used_at[ti] = step
        for ti in op["outputs"]:
            if ti >= 0:
                produced_at.setdefault(ti, step)

    graph_inputs = {d["index"] for d in interp.get_input_details()}
    graph_outputs = {d["index"] for d in interp.get_output_details()}
    for ti in graph_inputs:
        produced_at.setdefault(ti, -1)
    for ti in graph_outputs:
        last_used_at[ti] = len(ops)

    # Anything never produced by an op and not a graph input is a constant -> flash.
    activation_indices = set(produced_at)
    weight_bytes = sum(_tensor_bytes(by_index[i]) for i in by_index
                       if i not in activation_indices)

    peak, peak_step = 0, -1
    for step in range(len(ops)):
        live = 0
        for ti in activation_indices:
            if produced_at.get(ti, 1 << 30) <= step <= last_used_at.get(ti, -1):
                live += _tensor_bytes(by_index[ti])
        if live > peak:
            peak, peak_step = live, step

    naive = sum(_tensor_bytes(t) for t in tensors)
    return dict(
        method="peak_live_activation_set",
        n_ops=len(ops),
        peak_arena_bytes=int(peak),
        peak_arena_kb=peak / 1024,
        peak_at_op=int(peak_step),
        peak_op_name=ops[peak_step]["op_name"] if peak_step >= 0 else None,
        weights_in_flash_kb=weight_bytes / 1024,
        naive_sum_all_tensors_kb=naive / 1024,
        flash_kb=qr.size_kb,
        flash_headroom_kb=NRF52840_FLASH_KB - qr.size_kb,
        ram_headroom_kb=NRF52840_RAM_KB - peak / 1024,
        fits_nrf52840=bool(qr.size_kb < NRF52840_FLASH_KB and peak / 1024 < NRF52840_RAM_KB),
    )


def benchmark_latency(qr: QuantizationResult, X, RR_n, window_size, n_runs=200, warmup=20):
    """Single-window inference latency on *this* machine.

    This is a desktop/Colab x86 number and says nothing directly about the nRF52840's
    Cortex-M4 -- it is here to catch pathological regressions and to give a relative
    FP32-vs-INT8 figure. Real timing has to be measured on hardware.
    """
    interp = tf.lite.Interpreter(model_content=qr.tflite_bytes)
    in_det = interp.get_input_details()
    ecg_i, rr_i = _classify_inputs(in_det, window_size)
    interp.resize_tensor_input(in_det[ecg_i]["index"], [1, window_size, 1])
    interp.resize_tensor_input(in_det[rr_i]["index"], [1, RR_n.shape[1]])
    interp.allocate_tensors()
    in_det = interp.get_input_details()

    def _prep(arr, det):
        if det["dtype"] == np.int8:
            s, z = det["quantization"]
            return np.clip(np.round(arr / s + z), -128, 127).astype(np.int8)
        return arr.astype(np.float32)

    x = _prep(X[:1].reshape(1, window_size, 1), in_det[ecg_i])
    r = _prep(RR_n[:1], in_det[rr_i])

    for _ in range(warmup):
        interp.set_tensor(in_det[ecg_i]["index"], x)
        interp.set_tensor(in_det[rr_i]["index"], r)
        interp.invoke()

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        interp.set_tensor(in_det[ecg_i]["index"], x)
        interp.set_tensor(in_det[rr_i]["index"], r)
        interp.invoke()
        times.append((time.perf_counter() - t0) * 1000)
    times = np.array(times)
    return dict(n_runs=n_runs, mean_ms=float(times.mean()), p50_ms=float(np.percentile(times, 50)),
                p95_ms=float(np.percentile(times, 95)), min_ms=float(times.min()),
                note="host CPU, not nRF52840 -- measure on hardware for the real figure")


def int8_threshold_grid(qr: QuantizationResult, threshold):
    """Map a float threshold onto the INT8 output grid the device actually compares on.

    The sigmoid output is INT8, so probabilities land on ~256 discrete levels. Firmware
    that compares a dequantized float against a float threshold is doing the same
    comparison as `code > ceil(threshold/scale + zero_point) - 1`, only slower and with
    rounding risk. Bake the integer cutoff in instead, and report the *effective*
    threshold so the thesis quotes the operating point the device truly uses.
    """
    if not qr.full_int8:
        return dict(applicable=False, reason="model is not full-integer; output is float32")
    s, z = qr.output_scale, qr.output_zero_point
    code = int(np.floor(threshold / s + z))
    if (code - z) * s <= threshold:
        code += 1
    code = int(np.clip(code, -128, 127))
    return dict(applicable=True, float_threshold=float(threshold), output_scale=s,
                output_zero_point=z, int8_cutoff_code=code,
                effective_float_threshold=float((code - z) * s),
                n_distinct_levels=256,
                firmware_rule=f"classify as ARRHYTHMIA when raw int8 output >= {code}")


def export_c_header(tflite_bytes, header_path, array_name="tibok_model"):
    with open(header_path, "w") as f:
        f.write(f"#ifndef {array_name.upper()}_H\n#define {array_name.upper()}_H\n\n")
        f.write("#include <stddef.h>\n#ifndef __cplusplus\n#include <stdalign.h>\n#endif\n\n")
        f.write("#ifdef __cplusplus\nextern \"C\" {\n#endif\n\n")
        f.write(f"alignas(16) const unsigned char {array_name}[] = {{\n")
        for i, byte in enumerate(tflite_bytes):
            f.write(f"0x{byte:02x}, ")
            if (i + 1) % 12 == 0:
                f.write("\n")
        f.write(f"\n}};\n\nconst size_t {array_name}_len = {len(tflite_bytes)};\n\n")
        f.write("#ifdef __cplusplus\n}\n#endif\n\n#endif\n")
    print(f"Header written: {header_path} ({len(tflite_bytes)/1024:.2f} KB)")
    return header_path


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------

def quantize_and_test(model, X_val, RR_val_n, X_test, RR_test_n, y_test,
                      thresholds, window_size, run_tag="tibok", y_val=None,
                      test_symbols=None, deploy_threshold_name="precision_floor_90",
                      rr_mean=None, rr_std=None, n_calib=800, n_boot=2000,
                      batch_one=True, out_dir=".", allow_dynamic_range_fallback=False):
    """Full quantization + verification pass. Returns a JSON-serializable report."""
    os.makedirs(out_dir, exist_ok=True)
    report = {"run_tag": run_tag, "window_size": window_size}

    print("=" * 72)
    print("STEP 1 -- INT8 conversion")
    print("=" * 72)
    qr = quantize_int8(model, X_val, RR_val_n, window_size, n_calib=n_calib,
                       allow_dynamic_range_fallback=allow_dynamic_range_fallback)
    for a in qr.attempts:
        print(f"  {a}")
    print(f"\nStrategy: {qr.strategy}   full_int8={qr.full_int8}   size={qr.size_kb:.2f} KB")
    if not qr.full_int8:
        print("  *** WARNING: dynamic-range model -- will NOT run under TFLite-Micro. ***")
    report["quantization"] = qr.summary()

    tflite_path = os.path.join(out_dir, f"{run_tag}_model_int8.tflite")
    with open(tflite_path, "wb") as f:
        f.write(qr.tflite_bytes)

    print("\n" + "=" * 72)
    print("STEP 2 -- memory footprint vs nRF52840 budget")
    print("=" * 72)
    mem = estimate_tflm_arena(qr, window_size)
    report["memory"] = mem
    for k, v in mem.items():
        print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

    print("\n" + "=" * 72)
    print("STEP 3 -- inference (FP32 reference vs INT8), batch=1" if batch_one
          else "STEP 3 -- inference (FP32 reference vs INT8)")
    print("=" * 72)
    probs_fp32 = model.predict([X_test, RR_test_n], batch_size=256, verbose=0).flatten()
    t0 = time.time()
    probs_int8, raw_codes = run_tflite(qr, X_test, RR_test_n, window_size, batch_one=batch_one)
    print(f"  INT8 inference over {len(y_test)} test beats: {time.time()-t0:.1f}s")
    print(f"  FP32 ROC-AUC {roc_auc_score(y_test, probs_fp32):.4f} | "
          f"INT8 ROC-AUC {roc_auc_score(y_test, probs_int8):.4f}")
    if qr.full_int8:
        print(f"  Distinct INT8 output codes observed: {len(np.unique(raw_codes))} / 256")

    print("\n" + "=" * 72)
    print("STEP 4 -- paired FP32 vs INT8 comparison  (RQ2.1 / RQ2.3)")
    print("=" * 72)
    print("  Thresholds held fixed at their FP32 validation values -- the controlled")
    print("  comparison that answers the research question.")
    comparisons = {}
    for name, thr in thresholds.items():
        cmp = compare_fp32_int8(y_test, probs_fp32, probs_int8, thr, label=name, n_boot=n_boot)
        print_comparison(cmp)
        comparisons[name] = cmp
    report["comparisons"] = comparisons

    # --- Step 4b: what the device would actually do -------------------------------
    # Thresholds re-chosen on the INT8 model's own validation scores. See
    # `recalibrate_int8` for why the fixed-threshold numbers understate the shipped model.
    if y_val is not None:
        print("\n" + "=" * 72)
        print("STEP 4b -- INT8 with thresholds recalibrated on INT8 validation scores")
        print("=" * 72)
        int8_thr = recalibrate_int8(qr, model, X_val, RR_val_n, y_val, window_size,
                                    batch_one=batch_one)
        recal = {}
        print(f"{'operating point':<22}{'thr(FP32)':>11}{'thr(INT8)':>11}"
              f"{'F1':>9}{'prec':>9}{'sens':>9}{'spec':>9}")
        for name, thr in int8_thr.items():
            m = evaluate_at_threshold(y_test, probs_int8, thr, verbose=False)
            fixed = comparisons[name]["int8"]
            recal[name] = {"threshold_fp32": float(thresholds[name]),
                           "threshold_int8": float(thr), "metrics": m,
                           "delta_f1_vs_fixed": m["f1_score"] - fixed["f1_score"]}
            print(f"{name:<22}{thresholds[name]:>11.4f}{thr:>11.4f}"
                  f"{m['f1_score']:>9.4f}{m['precision']:>9.4f}"
                  f"{m['sensitivity']:>9.4f}{m['specificity']:>9.4f}")
        print("\n  recovery vs the fixed-threshold INT8 numbers above:")
        for name, d in recal.items():
            print(f"    {name:<22} F1 {d['delta_f1_vs_fixed']:+.4f}")
        report["int8_recalibrated"] = recal

    if test_symbols is not None:
        print("\n" + "=" * 72)
        print("STEP 5 -- per-AAMI-symbol sensitivity, FP32 vs INT8")
        print("=" * 72)
        thr = thresholds[deploy_threshold_name]
        sym = np.asarray(test_symbols)
        pf = (probs_fp32 > thr).astype(int)
        pq = (probs_int8 > thr).astype(int)
        breakdown = {}
        print(f"{'sym':<6}{'n':>7}{'FP32':>10}{'INT8':>10}{'delta':>9}")
        for s in sorted(set(sym[y_test == 1])):
            mask = (sym == s) & (y_test == 1)
            n = int(mask.sum())
            if not n:
                continue
            a, b = float(pf[mask].mean()), float(pq[mask].mean())
            breakdown[s] = dict(n=n, fp32_sensitivity=a, int8_sensitivity=b, delta=b - a)
            print(f"{s:<6}{n:>7}{a:>10.4f}{b:>10.4f}{b-a:>+9.4f}")
        report["symbol_breakdown"] = breakdown

    print("\n" + "=" * 72)
    print("STEP 6 -- latency + firmware constants")
    print("=" * 72)
    lat = benchmark_latency(qr, X_test, RR_test_n, window_size)
    report["latency"] = lat
    print(f"  mean {lat['mean_ms']:.3f} ms | p50 {lat['p50_ms']:.3f} | p95 {lat['p95_ms']:.3f}  ({lat['note']})")

    deploy_thr = thresholds[deploy_threshold_name]
    grid = int8_threshold_grid(qr, deploy_thr)
    report["int8_threshold"] = grid
    if grid.get("applicable"):
        print(f"\n  Deploy threshold {deploy_thr:.4f} -> int8 cutoff code {grid['int8_cutoff_code']} "
              f"(effective {grid['effective_float_threshold']:.4f})")
        print(f"  {grid['firmware_rule']}")

    header_path = os.path.join(out_dir, f"{run_tag}_model_int8.h")
    export_c_header(qr.tflite_bytes, header_path)

    fw = {
        "full_int8": qr.full_int8,
        "deployable_on_tflm": bool(qr.full_int8),
        "ecg_scale": qr.ecg_scale, "ecg_zero_point": qr.ecg_zero_point,
        "rr_scale": qr.rr_scale, "rr_zero_point": qr.rr_zero_point,
        "output_scale": qr.output_scale, "output_zero_point": qr.output_zero_point,
        "rr_feature_mean": list(map(float, rr_mean)) if rr_mean is not None else None,
        "rr_feature_std": list(map(float, rr_std)) if rr_std is not None else None,
        "deploy_threshold_name": deploy_threshold_name,
        "deploy_threshold_float": float(deploy_thr),
        "deploy_threshold_int8_code": grid.get("int8_cutoff_code"),
        "tensor_arena_bytes_estimate": mem.get("peak_arena_bytes"),
    }
    report["firmware"] = fw
    print("\n  Firmware constants:")
    for k, v in fw.items():
        print(f"    {k} = {v}")

    report_path = os.path.join(out_dir, f"{run_tag}_quantization_report.json")
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\nWrote: {tflite_path}\n       {header_path}\n       {report_path}")

    print("\n" + "=" * 72)
    print("VERDICT")
    print("=" * 72)
    d = comparisons[deploy_threshold_name]
    print(f"  Full-integer INT8: {'YES' if qr.full_int8 else 'NO -- NOT DEPLOYABLE'}")
    print(f"  Fits nRF52840:     {mem.get('fits_nrf52840')} "
          f"(flash {qr.size_kb:.1f}/{NRF52840_FLASH_KB} KB, "
          f"arena ~{mem.get('peak_arena_kb', 0):.1f}/{NRF52840_RAM_KB} KB)")
    print(f"  At {deploy_threshold_name}:  dF1 {d['delta_f1']['observed']:+.4f} "
          f"[{d['delta_f1']['ci_lo']:+.4f}, {d['delta_f1']['ci_hi']:+.4f}], "
          f"dSpec {d['delta_specificity']['observed']:+.4f} "
          f"[{d['delta_specificity']['ci_lo']:+.4f}, {d['delta_specificity']['ci_hi']:+.4f}]")
    print(f"  McNemar p={d['mcnemar']['p_value']:.4f} -> "
          f"{'quantization DID change predictions significantly' if d['mcnemar']['significant_at_0p05'] else 'no significant change from quantization'}")
    return report

In [ ]:
%%writefile tibok/data_paths.py
"""Locate the MIT-BIH / INCART record folders on a Colab Drive mount, and fail loudly.

The plain `os.walk(base_path)` indexing this replaces has one bad failure mode: `os.walk`
on a nonexistent path -- or on a `https://drive.google.com/...` sharing URL pasted in
where a filesystem path belongs -- does not raise. It yields nothing. The index comes
back empty, the "expect 75" warning scrolls past, and the run continues until it dies
much later inside `load_and_segment`, far from the actual mistake.

A Drive *sharing link* is a browser URL. Drive is mounted as a filesystem at
/content/drive, so the path is always /content/drive/MyDrive/... -- a URL can never be
one. If the folder was shared with you rather than owned by you, it does not appear under
MyDrive at all until you add a shortcut to it (Drive UI -> right-click the folder ->
"Organise" -> "Add shortcut to Drive"); only then does it show up as a normal directory.

So `resolve_db_path` rejects URLs outright, searches the mount for the records when the
configured path is wrong, and raises with the candidates it did find.
"""

from __future__ import annotations

import os
import re

__all__ = ["is_url", "index_records", "find_records_root", "resolve_db_path", "preflight",
           "DbSource", "resolve_db_source", "pick_lead"]

_URL_RE = re.compile(r"^[a-z][a-z0-9+.-]*://", re.I)

DEFAULT_SEARCH_ROOTS = ("/content/drive/MyDrive", "/content/drive/Shareddrives", "/content")


def is_url(s):
    return bool(_URL_RE.match(str(s).strip()))


def index_records(base_path):
    """Map record id -> extensionless path for every .hea under `base_path`."""
    index = {}
    for root, _dirs, files in os.walk(base_path):
        for f in files:
            if f.endswith(".hea"):
                index[f[:-4]] = os.path.join(root, f[:-4])
    return index


def find_records_root(probe_records, search_roots=DEFAULT_SEARCH_ROOTS, max_depth=6):
    """Find directories containing any of `probe_records` (e.g. ['I01', 'I75']).

    Returns [(directory, n_probe_records_found)] best first. Depth-capped because a full
    walk of a large Drive is slow and we only ever need the folder itself.
    """
    wanted = {f"{r}.hea" for r in probe_records}
    hits = {}
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        base_depth = root.rstrip("/").count("/")
        for dirpath, dirnames, files in os.walk(root):
            if dirpath.count("/") - base_depth >= max_depth:
                dirnames[:] = []
                continue
            found = wanted.intersection(files)
            if found:
                hits[dirpath] = max(hits.get(dirpath, 0), len(found))
    return sorted(hits.items(), key=lambda kv: -kv[1])


def resolve_db_path(configured, probe_records, label, search_roots=DEFAULT_SEARCH_ROOTS,
                    autodetect=True):
    """Return a usable directory for `label`, or raise with what was actually found."""
    configured = str(configured).strip()

    if is_url(configured):
        msg = (
            f"{label}: {configured!r} is a URL, not a filesystem path.\n"
            f"  Google Drive is MOUNTED as a filesystem, so the path always looks like\n"
            f"  /content/drive/MyDrive/<folder>. A browser sharing link is never a path --\n"
            f"  os.walk() on it silently returns nothing, which is why 0 records were found.\n"
            f"  If the folder was shared with you rather than owned by you, it will not be\n"
            f"  under MyDrive until you add a shortcut: in Drive, right-click the folder ->\n"
            f"  Organise -> 'Add shortcut to Drive'. Then use that shortcut's path here."
        )
        if autodetect:
            found = find_records_root(probe_records, search_roots)
            if found:
                best = found[0][0]
                print(f"{label}: configured path is a URL; auto-detected records at {best!r} instead.")
                for d, n in found[:5]:
                    print(f"    {d}  ({n}/{len(probe_records)} probe records)")
                return best
            msg += f"\n  Searched {list(search_roots)} for {probe_records} and found nothing."
        raise ValueError(msg)

    if os.path.isdir(configured):
        idx = index_records(configured)
        if any(r in idx for r in probe_records):
            return configured
        extra = ""
        if autodetect:
            found = find_records_root(probe_records, search_roots)
            if found:
                best = found[0][0]
                print(f"{label}: {configured!r} exists but holds none of {probe_records}; "
                      f"auto-detected {best!r} instead.")
                return best
            extra = f"\n  Searched {list(search_roots)} and found none of {probe_records}."
        raise FileNotFoundError(
            f"{label}: {configured!r} exists but contains none of {probe_records} "
            f"({len(idx)} .hea files found there).{extra}"
        )

    if autodetect:
        found = find_records_root(probe_records, search_roots)
        if found:
            best = found[0][0]
            print(f"{label}: {configured!r} does not exist; auto-detected {best!r} instead.")
            for d, n in found[:5]:
                print(f"    {d}  ({n}/{len(probe_records)} probe records)")
            return best

    raise FileNotFoundError(
        f"{label}: {configured!r} does not exist, and no folder containing "
        f"{probe_records} was found under {list(search_roots)}.\n"
        f"  Check the mount ran (drive.mount('/content/drive')) and that the folder is "
        f"under MyDrive -- a folder merely shared with you needs a shortcut first."
    )


def preflight(index, required, label, expected_total=None):
    """Verify every required record is present BEFORE loading starts.

    The old code printed a warning and carried on, so a missing database surfaced as a
    FileNotFoundError deep inside the loading loop, long after the real mistake. Checking
    up front means the error names the problem.
    """
    missing = [r for r in required if r not in index]
    print(f"{label}: {len(index)} .hea files indexed, {len(required) - len(missing)}/{len(required)} required records present")

    if expected_total is not None and len(index) != expected_total:
        print(
            f"  NOTE: expected about {expected_total} .hea files here but found {len(index)}. "
            f"That is not fatal -- only the {len(required)} required records are read -- but "
            f"an unexpected count usually means another database, a duplicate copy, or a "
            f"nested extraction is sharing this folder. Worth confirming it is what you think."
        )

    if missing:
        raise FileNotFoundError(
            f"{label}: {len(missing)} required records are missing: {missing}\n"
            f"  Indexed {len(index)} .hea files. Fix the path or the folder contents before "
            f"training -- continuing would change the study population."
        )
    return True


# ---------------------------------------------------------------------------
# Record sources: a local Drive folder, or PhysioNet streamed over HTTP
# ---------------------------------------------------------------------------

class DbSource:
    """Where a database's records are read from, and how to read one.

    Two kinds:

      local     -- records sit in a mounted directory (a Drive folder).
      physionet -- records are streamed per-record over HTTPS via wfdb's `pn_dir`.

    The PhysioNet kind exists because depending on a *shared* Drive folder is fragile in
    ways that have nothing to do with the code. View-only access is enough to read files,
    and "Add shortcut to Drive" works at view-only too, so permissions alone are usually
    not the blocker. But two things can still stop it:

      - If the owner ticked "Viewers cannot download, print, or copy", the FUSE mount
        cannot read the bytes at all, and no path fix helps.
      - Drive enforces per-file download quotas on widely-shared files. A run that reads
        75 multi-megabyte records can trip "download quota exceeded", which then blocks
        the folder for hours -- and it fails partway through a run, not at the start.

    Streaming from PhysioNet sidesteps both: INCART and MIT-BIH are open-access there, so
    the run depends on nothing anyone else controls. `wfdb.rdrecord(rec, pn_dir=...)`
    fetches one record at a time rather than mirroring the whole database, so there is no
    multi-hundred-megabyte download step.
    """

    def __init__(self, kind, label, path=None, index=None, pn_dir=None):
        self.kind = kind
        self.label = label
        self.path = path
        self.index = index or {}
        self.pn_dir = pn_dir

    def __repr__(self):
        where = self.path if self.kind == "local" else f"physionet:{self.pn_dir}"
        return f"<DbSource {self.label} {self.kind} {where!r}>"

    def read(self, record_id, ann_ext="atr"):
        """Return (record, annotation) for one record, from whichever source applies."""
        import wfdb
        if self.kind == "local":
            if record_id not in self.index:
                raise FileNotFoundError(
                    f"{self.label}: {record_id}.hea not found under {self.path!r}"
                )
            p = self.index[record_id]
            return wfdb.rdrecord(p), wfdb.rdann(p, ann_ext)
        return (wfdb.rdrecord(record_id, pn_dir=self.pn_dir),
                wfdb.rdann(record_id, ann_ext, pn_dir=self.pn_dir))


def resolve_db_source(configured, probe_records, label, pn_dir, required=None,
                      expected_total=None, search_roots=DEFAULT_SEARCH_ROOTS,
                      prefer="local"):
    """Resolve a database to a local folder, falling back to PhysioNet streaming.

    `prefer="physionet"` skips the local lookup entirely, which is the reproducible
    choice: it does not depend on anyone's Drive sharing settings.
    """
    if prefer == "physionet":
        print(f"{label}: using PhysioNet (pn_dir={pn_dir!r}) by request.")
        return DbSource("physionet", label, pn_dir=pn_dir)

    try:
        path = resolve_db_path(configured, probe_records, label, search_roots=search_roots)
    except (ValueError, FileNotFoundError) as e:
        print(f"{label}: local lookup failed, falling back to PhysioNet streaming.")
        print(f"    reason: {str(e).splitlines()[0]}")
        print(f"    reading from PhysioNet pn_dir={pn_dir!r} instead -- no Drive access needed.")
        return DbSource("physionet", label, pn_dir=pn_dir)

    index = index_records(path)
    if required is not None:
        try:
            preflight(index, required, label, expected_total=expected_total)
        except FileNotFoundError as e:
            print(f"{label}: local folder is incomplete, falling back to PhysioNet streaming.")
            print(f"    reason: {str(e).splitlines()[0]}")
            return DbSource("physionet", label, pn_dir=pn_dir)
    return DbSource("local", label, path=path, index=index)


def pick_lead(record, preferred, label="", record_id=""):
    """Index of the first `preferred` lead present in `record.sig_name`.

    Reading channel 0 and assuming it is MLII is wrong for MIT-BIH. Most records are
    ordered [MLII, V5], but not all -- record 114 is [V5, MLII], so channel 0 there is a
    completely different lead from every other record in the study. Selecting by name
    instead keeps the input channel consistent, which is the whole point of specifying a
    lead in the methodology.
    """
    names = [str(n).strip() for n in (record.sig_name or [])]
    for want in preferred:
        if want in names:
            return names.index(want)
    raise ValueError(
        f"{label} {record_id}: none of {list(preferred)} in sig_name={names}. "
        f"Refusing to guess a channel -- that would silently feed a different lead "
        f"into the model for this record."
    )

In [ ]:
%%writefile tibok/thresholds.py
"""Operating-point selection on the validation set.

Lifted out of the notebook so the trial harness can reuse it. Every threshold here is
chosen on VALIDATION and only ever applied to test -- that separation is the whole point,
and inlining this in the notebook made it easy to accidentally break when repeating a run.
"""

from __future__ import annotations

import numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_curve

__all__ = ["select_thresholds"]


def select_thresholds(y_val, val_probs, precision_floor=0.90, target_sensitivity=0.90):
    """Return the four operating points the study reports, all chosen on validation."""
    y_val = np.asarray(y_val)
    precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)

    f1 = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    f1_optimal = thresholds[np.argmax(f1[:-1])]

    specs, sens = [], []
    for t in thresholds:
        tn, fp, fn, tp = confusion_matrix(y_val, (val_probs > t).astype(int), labels=[0, 1]).ravel()
        specs.append(tn / (tn + fp) if (tn + fp) else 0.0)
        sens.append(tp / (tp + fn) if (tp + fn) else 0.0)
    specs, sens = np.array(specs), np.array(sens)

    youden = thresholds[np.argmax(sens + specs - 1)]

    mask = sens >= target_sensitivity
    high_sens = thresholds[np.argmax(np.where(mask, specs, -1))] if mask.any() else thresholds[np.argmax(sens)]

    # Lowest threshold still meeting the precision floor -- lowest keeps recall as high as
    # the floor allows.
    idx = np.where(precisions[:-1] >= precision_floor)[0]
    prec_floor = thresholds[idx[0]] if len(idx) else thresholds[np.argmax(precisions[:-1])]

    return {
        "f1_optimal": float(f1_optimal),
        "youden": float(youden),
        "high_sensitivity": float(high_sens),
        "precision_floor_90": float(prec_floor),
    }

In [ ]:
%%writefile tibok/trials.py
"""Repeat the whole train -> quantize -> evaluate experiment R times.

Why this is not just `N_CANDIDATES = 10`
----------------------------------------

Those two settings answer different questions, and only one of them answers RQ2.1/RQ2.3.

`N_CANDIDATES` is a *best-of-N search*. It trains N models, keeps whichever scores highest
on validation PR-AUC, and throws the rest away. Raising it does not give you more evidence
-- it gives you one model chosen from a larger pool, and it makes the winner's validation
PR-AUC *more* optimistically biased, because you are selecting the maximum of N noisy
draws and then reporting that maximum from the same set you selected on. The observed
spread in one run of this study (candidate 1 at 0.8865, candidate 2 at 0.7418) is a
0.14 PR-AUC gap from nothing but the seed, so with N=3 the "winner" is substantially luck.

A *trial* here is an independent replication: one seed, one model, its own
validation-chosen thresholds, its own INT8 conversion, its own test metrics. Running R of
them gives a distribution, so you can report "quantization costs F1 0.040 +/- 0.006 across
10 runs" instead of "quantization cost F1 0.040 in the one run we did". That is the
defensible form of the claim, and it is what an examiner asking "would you get this again"
is asking for.

The two compose: a trial may itself run a best-of-N search internally
(`candidates_per_trial > 1`), which is the honest way to keep the deployment-selection
procedure while still measuring its variability. Cost multiplies, so the default is 1.

What this does and does not measure
-----------------------------------

The patient split is FIXED across trials, deliberately -- it is the split the methodology
commits to, and re-drawing it per trial would change the study population and make trials
non-comparable. So the variance reported here is *training* variance: weight init,
augmentation draws, batch shuffling. It is not an estimate of how the result would move on
a different set of patients, which is a larger and separate question. Say which one you are
reporting.

Statistics
----------

Per-trial FP32-vs-INT8 deltas are paired by construction (same trial, same test beats), so
the summary runs a Wilcoxon signed-rank test over the R deltas. Do not instead pool every
beat from every trial into one big McNemar table: beats repeat across trials and the models
are correlated, so that inflates n and understates the p-value.
"""

from __future__ import annotations

import json
import os
import time

import numpy as np

from .quantization import (
    evaluate_at_threshold, quantize_int8, run_tflite, estimate_tflm_arena,
)
from .thresholds import select_thresholds

__all__ = ["run_trials", "summarize_trials", "print_trial_summary"]


def _metrics_row(y_test, probs, thresholds, prefix):
    out = {}
    for name, thr in thresholds.items():
        m = evaluate_at_threshold(y_test, probs, thr, verbose=False)
        for k in ("f1_score", "specificity", "sensitivity", "precision", "accuracy"):
            out[f"{prefix}_{name}_{k}"] = m[k]
        out[f"{prefix}_{name}_threshold"] = float(thr)
    out[f"{prefix}_roc_auc"] = float(
        evaluate_at_threshold(y_test, probs, list(thresholds.values())[0], verbose=False)["roc_auc"]
    )
    out[f"{prefix}_pr_auc"] = float(
        evaluate_at_threshold(y_test, probs, list(thresholds.values())[0], verbose=False)["pr_auc"]
    )
    return out


def run_trials(build_model, fit_model, X_train, RR_train, y_train,
               X_val, RR_val, y_val, X_test, RR_test, y_test,
               window_size, n_trials=10, base_seed=4000, candidates_per_trial=1,
               out_dir=".", run_tag="tibok", resume=True, batch_one=False,
               n_calib=800):
    """Run `n_trials` independent replications, checkpointing after each one.

    `build_model(seed)` must return a compiled model; `fit_model(model, seed)` trains it.
    Both are passed in rather than hard-coded so this module never has to import the
    notebook's architecture.

    Checkpointing matters here: 10 trials is over an hour of GPU time, and a Colab runtime
    that disconnects at trial 8 would otherwise lose everything. Results are appended to a
    JSON file after every trial and `resume=True` picks up where it stopped.
    """
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{run_tag}_trials.json")

    rows = []
    if resume and os.path.exists(path):
        rows = json.load(open(path))["trials"]
        print(f"Resuming: {len(rows)} trial(s) already recorded in {path}")

    done = {r["trial"] for r in rows}

    for t in range(n_trials):
        if t in done:
            continue
        seed = base_seed + t
        t0 = time.time()
        print(f"\n{'='*72}\nTRIAL {t + 1}/{n_trials}  (seed={seed})\n{'='*72}")

        best, best_val = None, -1.0
        for c in range(candidates_per_trial):
            cand_seed = seed * 100 + c
            model = build_model(cand_seed)
            fit_model(model, cand_seed)
            vp = model.predict([X_val, RR_val], batch_size=256, verbose=0).flatten()
            from sklearn.metrics import average_precision_score
            score = average_precision_score(y_val, vp)
            if candidates_per_trial > 1:
                print(f"  candidate {c + 1}/{candidates_per_trial}: val PR-AUC {score:.4f}")
            if score > best_val:
                best_val, best, best_vp = score, model, vp

        model, val_probs = best, best_vp
        thresholds = select_thresholds(y_val, val_probs)

        probs_fp32 = model.predict([X_test, RR_test], batch_size=256, verbose=0).flatten()
        qr = quantize_int8(model, X_val, RR_val, window_size, n_calib=n_calib, seed=seed)
        probs_int8, _ = run_tflite(qr, X_test, RR_test, window_size, batch_one=batch_one)

        row = {"trial": t, "seed": seed, "val_pr_auc_selected": float(best_val),
               "full_int8": bool(qr.full_int8), "strategy": qr.strategy,
               "size_kb": float(qr.size_kb),
               "arena_kb": float(estimate_tflm_arena(qr, window_size).get("peak_arena_kb", 0.0)),
               "wall_s": round(time.time() - t0, 1)}
        row.update(_metrics_row(y_test, probs_fp32, thresholds, "fp32"))
        row.update(_metrics_row(y_test, probs_int8, thresholds, "int8"))

        rows.append(row)
        json.dump({"run_tag": run_tag, "n_trials": n_trials, "trials": rows},
                  open(path, "w"), indent=2)
        f32 = row["fp32_precision_floor_90_f1_score"]
        i8 = row["int8_precision_floor_90_f1_score"]
        print(f"  trial {t + 1} done in {row['wall_s']:.0f}s | "
              f"FP32 F1 {f32:.4f} -> INT8 F1 {i8:.4f} ({i8 - f32:+.4f})")

    print(f"\nAll trials recorded in {path}")
    return rows


def summarize_trials(rows, threshold_name="precision_floor_90"):
    """Mean / SD / CI per metric, plus a Wilcoxon signed-rank on the per-trial deltas."""
    if not rows:
        raise ValueError("no trials to summarize")
    n = len(rows)
    summary = {"n_trials": n, "threshold": threshold_name, "metrics": {}}

    for metric in ("f1_score", "specificity", "sensitivity", "precision", "accuracy"):
        a = np.array([r[f"fp32_{threshold_name}_{metric}"] for r in rows])
        b = np.array([r[f"int8_{threshold_name}_{metric}"] for r in rows])
        d = b - a
        entry = {
            "fp32_mean": float(a.mean()), "fp32_sd": float(a.std(ddof=1)) if n > 1 else 0.0,
            "int8_mean": float(b.mean()), "int8_sd": float(b.std(ddof=1)) if n > 1 else 0.0,
            "delta_mean": float(d.mean()), "delta_sd": float(d.std(ddof=1)) if n > 1 else 0.0,
        }
        if n > 1:
            se = d.std(ddof=1) / np.sqrt(n)
            entry["delta_ci95"] = [float(d.mean() - 1.96 * se), float(d.mean() + 1.96 * se)]
            try:
                from scipy.stats import wilcoxon
                if np.any(d != 0):
                    entry["wilcoxon_p"] = float(wilcoxon(a, b).pvalue)
            except Exception:
                pass
        summary["metrics"][metric] = entry

    for k in ("val_pr_auc_selected", "size_kb", "arena_kb"):
        v = np.array([r[k] for r in rows], dtype=float)
        summary[k] = {"mean": float(v.mean()), "sd": float(v.std(ddof=1)) if n > 1 else 0.0,
                      "min": float(v.min()), "max": float(v.max())}
    summary["all_full_int8"] = bool(all(r["full_int8"] for r in rows))
    return summary


def print_trial_summary(summary):
    print(f"\n{'='*72}")
    print(f"TRIAL SUMMARY  (n={summary['n_trials']}, threshold={summary['threshold']})")
    print("=" * 72)
    print(f"{'metric':<14}{'FP32 mean±sd':>20}{'INT8 mean±sd':>20}{'delta mean±sd':>20}")
    for k, m in summary["metrics"].items():
        print(f"{k:<14}"
              f"{m['fp32_mean']:>12.4f}±{m['fp32_sd']:<7.4f}"
              f"{m['int8_mean']:>12.4f}±{m['int8_sd']:<7.4f}"
              f"{m['delta_mean']:>+12.4f}±{m['delta_sd']:<7.4f}")
    print()
    for k, m in summary["metrics"].items():
        if "delta_ci95" in m:
            p = m.get("wilcoxon_p")
            ptxt = f"  Wilcoxon p={p:.4f}" if p is not None else ""
            print(f"  {k:<14} delta 95% CI [{m['delta_ci95'][0]:+.4f}, {m['delta_ci95'][1]:+.4f}]{ptxt}")
    v = summary["val_pr_auc_selected"]
    print(f"\n  val PR-AUC across trials: {v['mean']:.4f} ± {v['sd']:.4f} "
          f"(min {v['min']:.4f}, max {v['max']:.4f})")
    print(f"  model size: {summary['size_kb']['mean']:.2f} KB  |  "
          f"arena: {summary['arena_kb']['mean']:.2f} KB  |  "
          f"all full-INT8: {summary['all_full_int8']}")

In [ ]:
%%writefile tibok/sweep.py
"""Sweep the loss/class-weight knobs that trade precision against sensitivity.

The study's two settings currently pull in opposite directions: `focal_loss(alpha=0.3)`
down-weights positives (0.3 against 0.7 for negatives) while `class_weight[1] *= 1.3`
pushes them back up, on top of an already-balanced weighting. The net is a mild recall
bias, arrived at by accident rather than by choice. Since the model is over on sensitivity
and under on precision, that bias is being paid for in exactly the wrong currency -- this
sweep measures the trade instead of guessing at it.

Selection hygiene
-----------------

A sweep that picks its winner on the test set makes the reported test metrics meaningless:
you would be choosing the configuration that happens to suit those particular patients, and
the number you quote would no longer estimate held-out performance. So nothing here touches
test data. The flow is:

  1. Split VALIDATION in two. `val_thr` chooses the operating point; `val_sel` scores the
     configuration. Without that split the comparison is circular -- you would pick a
     threshold on the same beats you then use to judge the threshold, which flatters every
     configuration and flatters the overfitted ones most.
  2. Rank configurations on `val_sel`.
  3. Take the winner to test EXACTLY ONCE, via `finalize_on_test`.

Step 3 is the only time test data is read, and it is a report, not a decision. If you find
yourself re-running the sweep after seeing test numbers, the test set has become a second
validation set and the honest move is to say so in the write-up.

Cost
----

One training run per (config, seed). A 6-config grid at one seed is roughly six training
runs. Seed noise in this study is large (0.8865 vs 0.7418 val PR-AUC across seeds), so a
one-seed sweep ranks configurations noisily; `n_seeds=2` or 3 averages that down at
proportional cost. Results checkpoint after every run and `resume=True` continues an
interrupted session.
"""

from __future__ import annotations

import json
import os
import time

import numpy as np

from .quantization import evaluate_at_threshold
from .thresholds import select_thresholds

__all__ = ["make_grid", "run_sweep", "summarize_sweep", "print_sweep", "finalize_on_test"]

DEFAULT_TARGETS = {"f1_score": 0.90, "precision": 0.90, "sensitivity": 0.95, "specificity": 0.95}


def make_grid(pos_boosts=(1.0, 1.3), alphas=(0.15, 0.30, 0.45), gammas=(3.0,)):
    """Grid over the knobs that actually move the precision/recall balance.

    `alpha` weights the positive term of the focal loss: lower favours precision, higher
    favours recall. `pos_boost` is the manual multiplier applied on top of balanced class
    weights -- 1.0 removes it entirely, which is the cleanest precision lever available.
    """
    return [{"pos_boost": p, "alpha": a, "gamma": g}
            for p in pos_boosts for a in alphas for g in gammas]


def _key(cfg, seed):
    return f"pb{cfg['pos_boost']}_a{cfg['alpha']}_g{cfg['gamma']}_s{seed}"


def run_sweep(train_one, configs, X_val, RR_val, y_val, window_size,
              n_seeds=1, base_seed=9000, val_split=0.5, out_dir=".", run_tag="tibok",
              resume=True, threshold_name="precision_floor_90", rng_seed=42):
    """Train one model per (config, seed) and score it on held-out validation.

    `train_one(config, seed)` must build, compile and fit a model, then return it. It is
    passed in rather than imported so this module never needs the notebook's architecture.
    """
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{run_tag}_sweep.json")

    rows = []
    if resume and os.path.exists(path):
        rows = json.load(open(path))["rows"]
        print(f"Resuming: {len(rows)} run(s) already recorded in {path}")
    done = {r["key"] for r in rows}

    y_val = np.asarray(y_val)
    rng = np.random.default_rng(rng_seed)
    idx = rng.permutation(len(y_val))
    cut = int(len(idx) * val_split)
    i_thr, i_sel = idx[:cut], idx[cut:]
    print(f"Validation split: {len(i_thr)} beats to choose the threshold, "
          f"{len(i_sel)} to score the configuration")

    total = len(configs) * n_seeds
    n = 0
    for cfg in configs:
        for s in range(n_seeds):
            seed = base_seed + s
            k = _key(cfg, seed)
            n += 1
            if k in done:
                continue
            t0 = time.time()
            print(f"\n[{n}/{total}] {k}  pos_boost={cfg['pos_boost']} "
                  f"alpha={cfg['alpha']} gamma={cfg['gamma']}")

            model = train_one(cfg, seed)
            probs = model.predict([X_val, RR_val], batch_size=256, verbose=0).flatten()

            thr = select_thresholds(y_val[i_thr], probs[i_thr])[threshold_name]
            m = evaluate_at_threshold(y_val[i_sel], probs[i_sel], thr, verbose=False)

            row = {"key": k, "seed": seed, **cfg, "threshold": float(thr),
                   "wall_s": round(time.time() - t0, 1)}
            row.update({f"val_{x}": m[x] for x in
                        ("f1_score", "precision", "sensitivity", "specificity",
                         "accuracy", "roc_auc", "pr_auc")})
            rows.append(row)
            json.dump({"run_tag": run_tag, "rows": rows}, open(path, "w"), indent=2)
            print(f"      val F1 {m['f1_score']:.4f} | prec {m['precision']:.4f} | "
                  f"sens {m['sensitivity']:.4f} | spec {m['specificity']:.4f}  "
                  f"({row['wall_s']:.0f}s)")

    print(f"\nSweep recorded in {path}")
    return rows


def summarize_sweep(rows, targets=None):
    """Average over seeds per config and mark which targets each configuration meets."""
    targets = targets or DEFAULT_TARGETS
    groups = {}
    for r in rows:
        k = (r["pos_boost"], r["alpha"], r["gamma"])
        groups.setdefault(k, []).append(r)

    out = []
    for (pb, a, g), rs in groups.items():
        e = {"pos_boost": pb, "alpha": a, "gamma": g, "n_seeds": len(rs)}
        for metric in ("f1_score", "precision", "sensitivity", "specificity", "pr_auc"):
            v = np.array([r[f"val_{metric}"] for r in rs])
            e[metric] = float(v.mean())
            e[f"{metric}_sd"] = float(v.std(ddof=1)) if len(v) > 1 else 0.0
        e["targets_met"] = sum(1 for m, t in targets.items() if e.get(m, 0) >= t)
        e["all_targets"] = e["targets_met"] == len(targets)
        # Shortfall against every target at once -- ranks by how far from passing, so a
        # config that misses one target narrowly beats one that misses two badly.
        e["shortfall"] = float(sum(max(0.0, t - e.get(m, 0)) for m, t in targets.items()))
        out.append(e)
    return sorted(out, key=lambda e: (e["shortfall"], -e["f1_score"]))


def print_sweep(summary, targets=None):
    targets = targets or DEFAULT_TARGETS
    print("\n" + "=" * 86)
    print("SWEEP  (validation only -- test set untouched)   ranked by distance to targets")
    print("=" * 86)
    print(f"{'pos_boost':>10}{'alpha':>7}{'gamma':>7}{'F1':>9}{'prec':>9}"
          f"{'sens':>9}{'spec':>9}{'met':>5}{'short':>8}")
    for e in summary:
        print(f"{e['pos_boost']:>10}{e['alpha']:>7}{e['gamma']:>7}"
              f"{e['f1_score']:>9.4f}{e['precision']:>9.4f}"
              f"{e['sensitivity']:>9.4f}{e['specificity']:>9.4f}"
              f"{e['targets_met']:>4}/{len(targets)}{e['shortfall']:>8.4f}")
    print(f"\n  targets: " + ", ".join(f"{m}>={t}" for m, t in targets.items()))
    best = summary[0]
    if best["all_targets"]:
        print(f"  -> {('pos_boost=%s alpha=%s' % (best['pos_boost'], best['alpha']))} "
              f"meets every target on validation.")
    else:
        miss = [m for m, t in targets.items() if best.get(m, 0) < t]
        print(f"  -> best config still misses: {', '.join(miss)}. "
              f"No threshold or loss setting fixes that -- it needs a better model.")


def finalize_on_test(model, X_test, RR_test, y_test, X_val, RR_val, y_val,
                     threshold_name="precision_floor_90", targets=None):
    """Score ONE chosen configuration on test. Call this once, after the sweep is settled.

    The threshold comes from the FULL validation set here (not the half-split used for
    ranking), because at this point there is nothing left to choose between -- the split
    existed only to keep the comparison honest.
    """
    targets = targets or DEFAULT_TARGETS
    val_probs = model.predict([X_val, RR_val], batch_size=256, verbose=0).flatten()
    thr = select_thresholds(y_val, val_probs)[threshold_name]
    probs = model.predict([X_test, RR_test], batch_size=256, verbose=0).flatten()
    m = evaluate_at_threshold(y_test, probs, thr, label=f"FINAL TEST ({threshold_name})")
    print("\n  target check:")
    for name, t in targets.items():
        v = m.get(name, 0.0)
        print(f"    {name:<14} {v:.4f}  vs  >={t:.2f}   {'PASS' if v >= t else 'FAIL'}")
    return m

In [ ]:
%%writefile tibok/balance.py
"""Balance the positive class BY SYMBOL, not just positive-vs-negative.

The problem
-----------

Two mechanisms in this study balance classes, and both are blind to which *kind* of
arrhythmia a beat is:

  - `augment_dataset(..., n_aug_positive=2)` copies every positive beat twice. V beats and
    F beats are boosted by the same factor, so the ratio between them is exactly preserved.
  - `class_weight` only ever sees the binary label, so it cannot tell a V from an F either.

The positive class is 97% V beats. So "balanced" training is, in practice, training a
V-beat detector, and the pooled sensitivity is a V-beat number wearing a label that says
"arrhythmia". One observed test split:

    V  6030 beats -> 0.974 sensitivity
    A   163 beats -> 0.859
    F    23 beats -> 0.087
    S     1 beat  -> 0.000

Sensitivity tracks sample count monotonically. Nothing about thresholds, quantization or
the loss function fixes that: the model was never given enough F beats to learn fusion
morphology, and the RR features actively mislead there, because a fusion beat is a sinus
beat and an ectopic beat arriving together and so is not especially premature.

Two levers
----------

`symbol_augmentation_plan` + `augment_by_symbol` oversample rare symbols more than common
ones. `symbol_sample_weights` instead reweights the loss per beat, which costs no memory
and creates no near-duplicates.

**The weights lever only works because `focal_loss` now returns one value per sample.**
While it reduced to a scalar, `sample_weight` had nothing per-sample to scale and was as
inert as `class_weight` was.

Strategies
----------

`"equal"` brings every positive symbol to the largest one's count. Honest in intent but
brutal in practice: with 700 F beats against 24k V beats it means ~34 copies of each F
beat, and the model can memorise those 700 rather than learn fusion.

`"sqrt"` (default) targets counts proportional to sqrt(n), the usual middle ground: rare
symbols gain a lot of weight without the common ones being drowned or the rare ones being
reduced to a few memorised examples.

`cap` bounds the multiplier regardless of strategy. Oversampling cannot create information
that is not in the data -- 23 F beats augmented 30x is still 23 F beats. If F sensitivity
matters for the claim, the real fix is more F beats, not more copies of these ones.
"""

from __future__ import annotations

import numpy as np

__all__ = ["symbol_counts", "symbol_augmentation_plan", "augment_by_symbol",
           "symbol_sample_weights"]


def symbol_counts(symbols, y):
    """Count positive beats per annotation symbol."""
    symbols = np.asarray(symbols)
    y = np.asarray(y)
    pos = symbols[y == 1]
    return {s: int((pos == s).sum()) for s in sorted(set(pos.tolist()))}


def symbol_augmentation_plan(symbols, y, strategy="sqrt", cap=10, min_count=0):
    """Return {symbol: extra_copies_per_beat} for the positive class.

    `extra_copies` is how many augmented duplicates to add per original beat, so 0 means
    "leave as is". Negative targets are clamped: this never discards data.
    """
    counts = symbol_counts(symbols, y)
    if not counts:
        return {}
    biggest = max(counts.values())

    plan = {}
    for s, n in counts.items():
        if n <= 0:
            continue
        if strategy == "equal":
            target = biggest
        elif strategy == "sqrt":
            # target ∝ sqrt(n), scaled so the largest symbol keeps its own count
            target = biggest * float(np.sqrt(n / biggest))
        elif strategy == "none":
            target = n
        else:
            raise ValueError(f"unknown strategy {strategy!r}")
        target = max(target, min_count)
        extra = max(0, int(round(target / n)) - 1)
        plan[s] = min(extra, cap)
    return plan


def augment_by_symbol(X, rr, y, symbols, augment_fn, rng, plan=None,
                      strategy="sqrt", cap=10, verbose=True):
    """Oversample positive beats per symbol according to `plan`.

    `augment_fn(segment, rng)` is the notebook's existing `augment_segment`. RR features are
    carried over unchanged for copies, since the augmentation perturbs the waveform and not
    beat timing -- the same assumption the original uniform augmentation made.
    """
    X = np.asarray(X)
    rr = np.asarray(rr)
    y = np.asarray(y)
    symbols = np.asarray(symbols)

    if plan is None:
        plan = symbol_augmentation_plan(symbols, y, strategy=strategy, cap=cap)

    Xs, rrs, ys, syms = [X], [rr], [y], [symbols]
    counts = symbol_counts(symbols, y)
    if verbose:
        print(f"{'sym':<6}{'n':>8}{'extra copies':>14}{'after':>10}")
    for s, extra in sorted(plan.items(), key=lambda kv: -counts.get(kv[0], 0)):
        n = counts.get(s, 0)
        if verbose:
            print(f"{s:<6}{n:>8}{extra:>14}{n * (1 + extra):>10}")
        if extra <= 0 or n == 0:
            continue
        idx = np.where((y == 1) & (symbols == s))[0]
        for _ in range(extra):
            Xs.append(np.stack([augment_fn(X[i], rng) for i in idx]).astype(np.float32))
            rrs.append(rr[idx])
            ys.append(np.ones(len(idx), dtype=y.dtype))
            syms.append(symbols[idx])

    return (np.concatenate(Xs), np.concatenate(rrs),
            np.concatenate(ys), np.concatenate(syms))


def symbol_sample_weights(symbols, y, strategy="sqrt", cap=10.0, negative_weight=1.0):
    """Per-beat loss weights that lift rare positive symbols.

    Pass the result as `sample_weight` to `model.fit`. Prefer this over oversampling when
    memory matters or when duplicating 23 beats 30 times feels like what it is.

    Requires a per-sample loss. With the old scalar-reducing focal loss these weights would
    be silently ignored, exactly as `class_weight` was.
    """
    symbols = np.asarray(symbols)
    y = np.asarray(y)
    counts = symbol_counts(symbols, y)
    if not counts:
        return np.full(len(y), negative_weight, dtype=np.float32)
    biggest = max(counts.values())

    w = np.full(len(y), float(negative_weight), dtype=np.float32)
    for s, n in counts.items():
        if strategy == "equal":
            wt = biggest / n
        elif strategy == "sqrt":
            wt = float(np.sqrt(biggest / n))
        elif strategy == "none":
            wt = 1.0
        else:
            raise ValueError(f"unknown strategy {strategy!r}")
        w[(y == 1) & (symbols == s)] = min(wt, cap)
    return w

## Data split — combined MIT-BIH + INCART, patient-level 70/15/15

Streams both databases directly from PhysioNet (no local mirror needed). MIT-BIH contributes
its 44 non-paced records; INCART contributes all 75 recordings. The pooled 119 "patients" are
shuffled once (fixed seed) and split 70/15/15 — train and test therefore each contain a mix of
both databases, so this protocol is **not** directly comparable to the classic de Chazal
DS1/DS2 MIT-BIH-only benchmark table in `TIBOK_RR_CNN.ipynb`; it's the protocol your own
methodology commits to, evaluated on a broader population.

In [ ]:
WINDOW_SIZE = 1250
FS = 360

MITDB_RECORDS = [
    '100', '101', '103', '105', '106', '108', '109', '111', '112', '113',
    '114', '115', '116', '117', '118', '119', '121', '122', '123', '124',
    '200', '201', '202', '203', '205', '207', '208', '209', '210', '212',
    '213', '214', '215', '219', '220', '221', '222', '223', '228', '230',
    '231', '232', '233', '234'
]  # MIT-BIH, paced-beat records 102/104/107/217 excluded
INCART_RECORDS = [f"I{i:02d}" for i in range(1, 76)]  # all 75 St. Petersburg INCART recordings

ALL_RECORDS = [('mitdb', r) for r in MITDB_RECORDS] + [('incartdb', r) for r in INCART_RECORDS]

rng_split = np.random.RandomState(42)
shuffled = list(ALL_RECORDS)
rng_split.shuffle(shuffled)
n_total = len(shuffled)
n_train = round(0.70 * n_total)
n_val = round(0.15 * n_total)
TRAIN_RECORDS = shuffled[:n_train]
VAL_RECORDS = shuffled[n_train:n_train + n_val]
TEST_RECORDS = shuffled[n_train + n_val:]

print(f"Total patient-records: {n_total} (MIT-BIH {len(MITDB_RECORDS)} + INCART {len(INCART_RECORDS)})")
print(f"Train ({len(TRAIN_RECORDS)}): {TRAIN_RECORDS}")
print(f"Val   ({len(VAL_RECORDS)}): {VAL_RECORDS}")
print(f"Test  ({len(TEST_RECORDS)}): {TEST_RECORDS}")

LABEL_MAP = {
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
    'V': 1, 'A': 1, 'F': 1, 'S': 1, 'a': 1, 'J': 1,
}
BEAT_SYMBOLS = set(LABEL_MAP.keys()) | {'B', 'r', 'n', 'E', 'f', 'Q', '/'}

assert not any(r in TRAIN_RECORDS for r in VAL_RECORDS)
assert not any(r in TRAIN_RECORDS for r in TEST_RECORDS)
assert not any(r in VAL_RECORDS for r in TEST_RECORDS)
print("Data split integrity OK.")

## Local Drive cache for MIT-BIH + INCART

Both databases are read directly from Google Drive — no download/streaming step.

**Set these to filesystem paths, not sharing links.** Drive is *mounted* at
`/content/drive`, so a folder's path always looks like `/content/drive/MyDrive/<folder>`.
A `https://drive.google.com/drive/folders/...` URL is a browser link and is never a valid
path — and `os.walk()` on one does not raise, it just yields nothing, so the index comes
back empty and the failure surfaces much later and far from its cause.

If a folder was *shared with you* rather than owned by you, it does not appear under
`MyDrive` at all until you add a shortcut: in Drive, right-click the folder → Organise →
**Add shortcut to Drive**. Then use that shortcut's path here.

`resolve_db_source` enforces this: it rejects URLs, searches the mount for the records
when the configured path is wrong, and verifies every required record is present *before*
loading starts.

**If Drive is not usable, it streams from PhysioNet instead** — set `INCART_PREFER` /
`MITDB_PREFER` to `"physionet"` to force that, or leave them `"local"` and the fallback
happens automatically when the folder is missing or incomplete. Both databases are
open-access on PhysioNet and `wfdb` fetches one record at a time, so there is no bulk
download step.

That fallback matters for a *shared* folder. View-only access is enough to read files, and
"Add shortcut to Drive" works at view-only too — but if the owner ticked **"Viewers cannot
download, print, or copy"**, the mount cannot read the bytes at all, and Drive also enforces
per-file download quotas on widely-shared files that a 75-record run can trip partway
through. PhysioNet depends on none of that.

In [ ]:
from tibok.data_paths import resolve_db_source, pick_lead

MITDB_PATH = "/content/drive/MyDrive/mit-bih-arrhythmia-database-1.0.0"
INCART_PATH = "/content/drive/MyDrive/incart-arrhythmia-database-1.0.0"  # <-- a PATH, not a sharing URL

# Set either of these to "physionet" to skip Drive entirely for that database and stream
# the records over HTTPS instead. That is the reproducible option: it depends on nothing
# anyone's sharing settings control, and wfdb fetches one record at a time rather than
# mirroring the whole database. Left as "local", a Drive folder is used when it is present
# and complete, and PhysioNet is used automatically when it is not.
MITDB_PREFER = "local"
INCART_PREFER = "local"

DB = {
    "mitdb": resolve_db_source(MITDB_PATH, ["100", "234"], "MIT-BIH", pn_dir="mitdb",
                               required=MITDB_RECORDS, expected_total=48, prefer=MITDB_PREFER),
    "incartdb": resolve_db_source(INCART_PATH, ["I01", "I75"], "INCART", pn_dir="incartdb",
                                  required=INCART_RECORDS, expected_total=75, prefer=INCART_PREFER),
}
for key, src in DB.items():
    print(f"  {key}: {src}")

## Loading + RR-interval feature extraction (both databases, read from Drive)

Same RR-interval logic as before, generalized to handle both sources: MIT-BIH loads at its
native 360 Hz directly from `MITDB_PATH`; INCART loads from `INCART_PATH` and is resampled from
257 Hz to 360 Hz (`scipy.signal.resample`), with its annotation sample indices rescaled to
match, so every downstream step (windowing, RR-interval computation in seconds) operates in one
consistent 360 Hz time base regardless of source. INCART's lead II is used as the
MLII-equivalent channel, per the document's stated data-collection plan.

In [ ]:
import scipy.signal

# Which lead to pull per database. Selecting by NAME matters: MIT-BIH is mostly ordered
# [MLII, V5], but record 114 is [V5, MLII], so the old `p_signal[:, 0]` fed V5 into the
# model for that one record while every other record contributed MLII. `pick_lead` raises
# rather than guessing when no preferred lead is present.
LEAD_PREFERENCE = {'mitdb': ('MLII', 'II'), 'incartdb': ('II',)}


def load_and_segment(source, record_id, window_size):
    record, annotation = DB[source].read(record_id)
    lead_idx = pick_lead(record, LEAD_PREFERENCE[source], label=source, record_id=record_id)
    signal = record.p_signal[:, lead_idx]
    native_fs = record.fs  # MIT-BIH 360 Hz, INCART 257 Hz

    beat_mask = np.array([s in BEAT_SYMBOLS for s in annotation.symbol])
    beat_samples_native = annotation.sample[beat_mask]
    beat_symbols = np.array(annotation.symbol)[beat_mask]
    if len(beat_samples_native) < 3:
        return [], [], [], []

    if native_fs != FS:
        n_new = int(round(len(signal) * FS / native_fs))
        signal = scipy.signal.resample(signal, n_new)
        beat_samples = np.round(beat_samples_native * (FS / native_fs)).astype(int)
    else:
        beat_samples = beat_samples_native

    half = window_size // 2
    sig_len = len(signal)

    rr = np.diff(beat_samples) / FS
    pre_rr = np.empty(len(beat_samples)); post_rr = np.empty(len(beat_samples))
    pre_rr[0] = rr[0] if len(rr) else 0.8; pre_rr[1:] = rr
    post_rr[-1] = rr[-1] if len(rr) else 0.8; post_rr[:-1] = rr

    local_rr = np.empty(len(beat_samples))
    global_mean_rr = rr.mean() if len(rr) else 0.8
    for i in range(len(beat_samples)):
        window = pre_rr[max(0, i - 10):i]
        local_rr[i] = window.mean() if len(window) > 0 else global_mean_rr
    ratio = pre_rr / (local_rr + 1e-6)

    segments, labels, symbols, rr_feats = [], [], [], []
    for i, (sample, symbol) in enumerate(zip(beat_samples, beat_symbols)):
        if symbol not in LABEL_MAP:
            continue
        if sample - half < 0 or sample + half > sig_len:
            continue
        segments.append(signal[sample - half: sample + half])
        labels.append(LABEL_MAP[symbol])
        symbols.append(symbol)
        rr_feats.append([pre_rr[i], post_rr[i], local_rr[i], ratio[i]])
    return segments, labels, symbols, rr_feats


def build_dataset(record_list):
    all_segments, all_labels, all_symbols, all_rr = [], [], [], []
    for source, rid in record_list:
        segs, labs, syms, rrf = load_and_segment(source, rid, WINDOW_SIZE)
        all_segments.extend(segs); all_labels.extend(labs)
        all_symbols.extend(syms); all_rr.extend(rrf)
        print(f"  {source}/{rid}: {len(segs)} beats loaded")
    return all_segments, all_labels, all_symbols, all_rr


def normalize_batch(segments):
    arr = np.stack(segments).astype(np.float32)
    mean = arr.mean(axis=1, keepdims=True)
    std = arr.std(axis=1, keepdims=True) + 1e-8
    return (arr - mean) / std


print("Loading training records...")
train_segments, train_labels, train_symbols, train_rr = build_dataset(TRAIN_RECORDS)
print("\nLoading validation records...")
val_segments, val_labels, val_symbols, val_rr = build_dataset(VAL_RECORDS)
print("\nLoading held-out test records...")
test_segments, test_labels, test_symbols, test_rr = build_dataset(TEST_RECORDS)
y_test_symbols = np.array(test_symbols)

X_train = normalize_batch(train_segments).reshape(-1, WINDOW_SIZE, 1)
y_train = np.array(train_labels)
RR_train = np.array(train_rr, dtype=np.float32)

X_val = normalize_batch(val_segments).reshape(-1, WINDOW_SIZE, 1)
y_val = np.array(val_labels)
RR_val = np.array(val_rr, dtype=np.float32)

X_test = normalize_batch(test_segments).reshape(-1, WINDOW_SIZE, 1)
y_test = np.array(test_labels)
RR_test = np.array(test_rr, dtype=np.float32)

print(f"\nX_train {X_train.shape}  X_val {X_val.shape}  X_test {X_test.shape}")
print("Train class balance:", pd.Series(y_train).value_counts(normalize=True).to_dict())
print("Test  class balance:", pd.Series(y_test).value_counts(normalize=True).to_dict())

rr_mean = RR_train.mean(axis=0)
rr_std = RR_train.std(axis=0) + 1e-8
RR_train_n = (RR_train - rr_mean) / rr_std
RR_val_n = (RR_val - rr_mean) / rr_std
RR_test_n = (RR_test - rr_mean) / rr_std

## Minority-class augmentation — balanced BY SYMBOL, not just positive-vs-negative

Same signal recipe as before (time shift, baseline wander, powerline hum, Gaussian noise,
amplitude scaling), with RR features carried over unchanged for copies since the
augmentation perturbs the waveform and not beat timing.

**What changed: the balancing is now symbol-aware.** Copying every positive beat 2x boosts
V and F by the same factor, so it preserves the imbalance *inside* the positive class
exactly. That class is ~97% V beats, so "balanced" training was in practice training a
V-beat detector, and the pooled sensitivity was a V-beat number wearing an "arrhythmia"
label. One observed test split:

    V  6030 beats -> 0.974 sensitivity
    A   163 beats -> 0.859
    F    23 beats -> 0.087
    S     1 beat  -> 0.000

Sensitivity tracks sample count monotonically. No threshold, loss or quantization setting
fixes that — the model was never shown enough F beats, and the RR features actively mislead
there, since a fusion beat is a sinus and an ectopic beat arriving together and so is not
especially premature.

`BALANCE_STRATEGY` controls it: `"sqrt"` (default) targets counts proportional to sqrt(n),
`"equal"` levels every symbol to the largest, `"none"` reproduces the old uniform 2x
behaviour. `BALANCE_CAP` bounds the multiplier.

**Oversampling cannot create information that is not in the data.** 23 F beats copied 30
times are still 23 F beats, and the model may simply memorise them. If F sensitivity has to
hold up in the write-up, the honest fix is more F beats — or reporting F separately and
saying the study is underpowered for it.

In [ ]:
def augment_segment(segment, rng, shift_max=40, noise_std=0.03, scale_range=(0.9, 1.1),
                     baseline_wander_prob=0.3, powerline_prob=0.3):
    seg = segment.copy().astype(np.float32).flatten()
    n = len(seg)
    shift = int(rng.integers(-shift_max, shift_max + 1))
    seg = np.roll(seg, shift)
    if rng.random() < baseline_wander_prob:
        freq = rng.uniform(0.15, 0.4); t = np.arange(n)
        wander_amp = rng.uniform(0.05, 0.15)
        seg = seg + wander_amp * np.sin(2 * np.pi * freq * t / n)
    if rng.random() < powerline_prob:
        freq = rng.choice([50, 60]); t = np.arange(n)
        pl_amp = rng.uniform(0.02, 0.06)
        seg = seg + pl_amp * np.sin(2 * np.pi * freq * t / FS)
    seg = seg + rng.normal(0, noise_std, n).astype(np.float32)
    seg = seg * rng.uniform(*scale_range)
    return seg.reshape(segment.shape)


def augment_dataset(X, rr, y, rng, n_aug_positive=2):
    X_list, rr_list, y_list = [X], [rr], [y]
    idx = np.where(y == 1)[0]
    for _ in range(n_aug_positive):
        X_aug = np.stack([augment_segment(X[i], rng) for i in idx]).astype(np.float32)
        rr_aug = rr[idx]
        y_aug = np.full(len(idx), 1)
        X_list.append(X_aug); rr_list.append(rr_aug); y_list.append(y_aug)
    return np.concatenate(X_list), np.concatenate(rr_list), np.concatenate(y_list)


from tibok.balance import augment_by_symbol, symbol_counts, symbol_sample_weights

BALANCE_STRATEGY = "sqrt"   # "sqrt" | "equal" | "none" (none == the old uniform 2x)
BALANCE_CAP = 10            # ceiling on extra copies per beat

aug_rng = np.random.default_rng(42)
train_symbols_arr = np.array(train_symbols)

print("Positive-class composition BEFORE balancing:", symbol_counts(train_symbols_arr, y_train))
X_train_aug, RR_train_aug, y_train_aug, sym_train_aug = augment_by_symbol(
    X_train, RR_train_n, y_train, train_symbols_arr,
    augment_segment, aug_rng, strategy=BALANCE_STRATEGY, cap=BALANCE_CAP,
)
print("Positive-class composition AFTER  balancing:", symbol_counts(sym_train_aug, y_train_aug))

shuffle_idx = np.random.RandomState(42).permutation(len(y_train_aug))
X_train_aug = X_train_aug[shuffle_idx]; RR_train_aug = RR_train_aug[shuffle_idx]
y_train_aug = y_train_aug[shuffle_idx]; sym_train_aug = sym_train_aug[shuffle_idx]
print(f"After augmentation: X_train {X_train_aug.shape}, binary balance:",
      pd.Series(y_train_aug).value_counts(normalize=True).to_dict())

# Alternative/complementary lever: reweight the loss per beat instead of duplicating data.
# This only works now that focal_loss returns one value per sample -- with the old
# scalar-reducing version, sample_weight was as inert as class_weight was.
USE_SYMBOL_SAMPLE_WEIGHTS = False
sample_weights = (symbol_sample_weights(sym_train_aug, y_train_aug, strategy=BALANCE_STRATEGY)
                  if USE_SYMBOL_SAMPLE_WEIGHTS else None)

## Model: CNN branch (morphology) + RR branch (rhythm timing), fused before the classifier head

Uses the same fixed (both-term) focal loss as the baseline notebook, plus a manually boosted
positive-class weight — this is a screening device, so a false negative (missed arrhythmia)
should cost more than a false positive.

In [ ]:
def focal_loss(gamma=3.0, alpha=0.3):
    """Focal loss, returning ONE VALUE PER SAMPLE.

    The previous version ended with `tf.reduce_mean(tf.reduce_sum(..., axis=-1))`, which
    collapses the batch axis and hands Keras a scalar. `class_weight` works by scaling each
    sample's loss, so with nothing per-sample left to scale it was almost entirely inert:
    on a 15%-positive problem, a 10x positive weight moved the mean prediction by +0.0036
    with the scalar form against +0.0639 with this one. Every class-weight setting in this
    notebook -- the balanced weights and the 1.3x recall boost on top of them -- was
    therefore doing close to nothing.

    Keeping the batch axis and letting Keras reduce is the fix; the maths is unchanged.
    """
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        pos_term = -alpha * y_true * tf.pow(1.0 - y_pred, gamma) * tf.math.log(y_pred)
        neg_term = -(1.0 - alpha) * (1.0 - y_true) * tf.pow(y_pred, gamma) * tf.math.log(1.0 - y_pred)
        return tf.reduce_sum(pos_term + neg_term, axis=-1)
    return focal_loss_fixed


def build_model(window_size, n_rr_features=4):
    sig_in = layers.Input(shape=(window_size, 1), name="ecg_window")
    x = sig_in
    for filters, kernel, stride in [(16, 7, 4), (32, 7, 4), (64, 7, 2)]:
        x = layers.Conv1D(filters, kernel_size=kernel, strides=stride, padding='same',
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.SpatialDropout1D(0.15)(x)
    x = layers.GlobalAveragePooling1D()(x)

    rr_in = layers.Input(shape=(n_rr_features,), name="rr_features")
    r = layers.Dense(16, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(rr_in)

    merged = layers.Concatenate()([x, r])
    merged = layers.Dropout(0.5)(merged)
    merged = layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(merged)
    merged = layers.Dropout(0.4)(merged)
    out = layers.Dense(1, activation='sigmoid')(merged)

    model = Model(inputs=[sig_in, rr_in], outputs=out)
    model.compile(
        optimizer=Adam(learning_rate=3e-4),
        loss=focal_loss(gamma=3.0, alpha=0.3),
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model


build_model(WINDOW_SIZE).summary()

## Training — best-of-N candidate search, model selected by validation PR-AUC

On Colab with a GPU this trains in a couple of minutes; raise `N_CANDIDATES` (e.g. to 8, as in
the baseline notebook) for a more thorough search once you've confirmed the pipeline runs
end-to-end.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_aug), y=y_train_aug)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
class_weight_dict[1] *= 1.3  # extra recall bias — a missed arrhythmia is costlier than a false alarm
print(f"Class weights: {class_weight_dict}")

N_CANDIDATES = 3
EPOCHS = 50
BATCH_SIZE = 128

best_val_pr_auc = -1
best_model = None
all_candidates, candidate_val_pr_aucs = [], []

t0 = time.time()
for i in range(N_CANDIDATES):
    print(f"\n=== Candidate {i+1}/{N_CANDIDATES} ===")
    tf.keras.utils.set_random_seed(4000 + i)
    candidate = build_model(WINDOW_SIZE)
    candidate.fit(
        [X_train_aug, RR_train_aug], y_train_aug,
        validation_data=([X_val, RR_val_n], y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        class_weight=class_weight_dict, sample_weight=sample_weights, verbose=2,
    )
    val_probs = candidate.predict([X_val, RR_val_n], batch_size=256, verbose=0).flatten()
    val_pr_auc = average_precision_score(y_val, val_probs)
    print(f"Candidate {i+1} val PR-AUC: {val_pr_auc:.4f}")
    all_candidates.append(candidate)
    candidate_val_pr_aucs.append(val_pr_auc)
    if val_pr_auc > best_val_pr_auc:
        best_val_pr_auc = val_pr_auc
        best_model = candidate

print(f"\nTraining wall time: {(time.time()-t0)/60:.1f} min")
print(f"Best candidate val PR-AUC: {best_val_pr_auc:.4f}  (per-candidate: {candidate_val_pr_aucs})")
model = best_model
model.save(f'{RUN_TAG}_model.keras')

## Threshold selection on validation, evaluation on the held-out DS2 test set

Reports three operating points, same as the baseline notebook: F1-optimal, Youden's-J
(balanced sensitivity/specificity), and a high-sensitivity point targeting ≥90% recall. Given
TIBOK is a *screening* device — the hypotheses explicitly treat a missed arrhythmia as
costlier than a false alarm — the Youden or high-sensitivity threshold is the more defensible
deployment choice, not F1-optimal.

In [ ]:
y_val_probs = model.predict([X_val, RR_val_n], batch_size=256, verbose=0).flatten()
precisions_val, recalls_val, thresholds_val = precision_recall_curve(y_val, y_val_probs)
f1_scores_val = 2 * (precisions_val * recalls_val) / (precisions_val + recalls_val + 1e-8)
f1_optimal_threshold = thresholds_val[np.argmax(f1_scores_val[:-1])]

specs, sens = [], []
for t in thresholds_val:
    preds = (y_val_probs > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    sens.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
specs, sens = np.array(specs), np.array(sens)
youden_threshold = thresholds_val[np.argmax(sens + specs - 1)]

TARGET_SENSITIVITY = 0.90
valid_mask = sens >= TARGET_SENSITIVITY
if valid_mask.any():
    valid_specs = np.where(valid_mask, specs, -1)
    high_sens_threshold = thresholds_val[np.argmax(valid_specs)]
else:
    high_sens_threshold = thresholds_val[np.argmax(sens)]

# Precision-floor threshold: the lowest threshold on VALIDATION that still reaches
# >=90% precision there (picking the lowest such threshold keeps recall as high as
# possible while satisfying the floor). Chosen on val, only ever evaluated on test —
# same no-leakage protocol as the other three thresholds. This has empirically been
# the best-performing operating point (higher precision AND higher F1 than F1-optimal,
# for only a small recall cost) on both models trained so far, so it's the deploy
# recommendation below — but re-check it after every retrain, since the "best"
# threshold depends on that run's specific precision-recall curve.
PRECISION_FLOOR_TARGET = 0.90
valid_prec_idx = np.where(precisions_val[:-1] >= PRECISION_FLOOR_TARGET)[0]
if len(valid_prec_idx) > 0:
    precision_floor_90_threshold = thresholds_val[valid_prec_idx[0]]
else:
    precision_floor_90_threshold = thresholds_val[np.argmax(precisions_val[:-1])]

print(f"F1-optimal threshold (val): {f1_optimal_threshold:.4f}")
print(f"Youden's-J threshold (val): {youden_threshold:.4f}")
print(f"High-sensitivity threshold (val, target>=90%): {high_sens_threshold:.4f}")
print(f"Precision-floor threshold (val, target>=90% precision): {precision_floor_90_threshold:.4f}")


def evaluate(y_true, probs, threshold, label):
    pred = (probs > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    metrics = dict(
        threshold=float(threshold), accuracy=accuracy_score(y_true, pred),
        precision=precision_score(y_true, pred, zero_division=0),
        sensitivity=recall_score(y_true, pred, zero_division=0),
        specificity=tn / (tn + fp) if (tn + fp) > 0 else 0,
        f1_score=f1_score(y_true, pred, zero_division=0),
        roc_auc=roc_auc_score(y_true, probs), pr_auc=average_precision_score(y_true, probs),
        tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
    )
    print(f"\n--- {label} (threshold={threshold:.4f}) ---")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    return metrics


y_test_probs = model.predict([X_test, RR_test_n], batch_size=256, verbose=0).flatten()

results = {
    "f1_optimal": evaluate(y_test, y_test_probs, f1_optimal_threshold, "TEST — F1-optimal"),
    "youden": evaluate(y_test, y_test_probs, youden_threshold, "TEST — Youden-balanced"),
    "high_sensitivity": evaluate(y_test, y_test_probs, high_sens_threshold, "TEST — High-sensitivity (>=90% target)"),
    "precision_floor_90": evaluate(y_test, y_test_probs, precision_floor_90_threshold, "TEST — Precision-floor (>=90% val precision) [recommended deploy threshold]"),
}

## Per-AAMI-symbol sensitivity breakdown

The pooled "arrhythmia" sensitivity can hide a real weakness — if the model is strong on
common V-beats but weak on rarer F or J beats, the pooled number still looks fine because
V-beats dominate the positive class.

In [ ]:
y_test_pred = (y_test_probs > precision_floor_90_threshold).astype(int)
symbol_breakdown = {}
for sym in sorted(set(y_test_symbols[y_test == 1])):
    mask = (y_test_symbols == sym) & (y_test == 1)
    n = int(mask.sum())
    if n == 0:
        continue
    caught = int((y_test_pred[mask] == 1).sum())
    symbol_breakdown[sym] = {"n": n, "caught": caught, "sensitivity": caught / n}

print("Per-AAMI-symbol sensitivity (precision-floor threshold):")
for sym, d in symbol_breakdown.items():
    print(f"  {sym}: {d['caught']}/{d['n']} ({d['sensitivity']:.4f})")

## INT8 post-training quantization — the model that actually ships on the nRF52840

Calibrates on the validation set (never the test set), converts both inputs to INT8, and
re-evaluates on the same held-out test set to answer RQ2.1/RQ2.3 (does quantization change
F1 / specificity?).

**The `input->dims->size != 4 (3 != 4)` failure is fixed, and it was not what the earlier note
in this notebook said it was.** It is neither environment-specific nor weight-dependent — a
freshly-initialized model fails identically once you actually run the calibration pass. The
cause is that `TFLiteConverter` does not preserve `model.inputs` ordering for a multi-input
model: on TF 2.21 it emits the graph inputs as `[rr_features, ecg_window]` while Keras lists
them as `[ecg_window, rr_features]`. The old `representative_dataset` yielded
`[X_val_sample, RR_val_sample]` positionally, so the 4-element RR vector was fed into the
Conv1D branch — rank 3 where the kernel wants rank 4 — hence `3 != 4` at CONV_2D node 1.

The `tibok.quantization` module (written to disk by the cell above) fixes this by probing the converter's actual input order first (one
throwaway float conversion) and then feeding calibration samples in *that* order. It also no
longer silently falls back to dynamic-range quantization: TFLite-Micro has no dynamic-range
kernels for this graph, so that fallback produced a model that loads on desktop but fails at
`AllocateTensors()` on the nRF52840 — a conversion bug quietly converted into a firmware bug.

In [ ]:
import sys

# In the generated Colab notebook the cell above this one is a `%%writefile` cell that
# drops `tibok/quantization.py` straight onto the runtime's disk, so the notebook is
# self-contained: no clone, no upload, no GitHub auth. That matters because this repo is
# private -- `git clone` from a Colab runtime would prompt for credentials and fail.
#
# Running this file as a plain script instead (outside Colab) just imports the module
# from the repo checkout it already sits in.
if "." not in sys.path:
    sys.path.insert(0, ".")
from tibok.quantization import quantize_and_test

THRESHOLDS = {
    "f1_optimal": float(f1_optimal_threshold),
    "youden": float(youden_threshold),
    "high_sensitivity": float(high_sens_threshold),
    "precision_floor_90": float(precision_floor_90_threshold),
}

quant_report = quantize_and_test(
    model=model,
    X_val=X_val, RR_val_n=RR_val_n,
    X_test=X_test, RR_test_n=RR_test_n, y_test=y_test,
    y_val=y_val,   # enables STEP 4b: thresholds recalibrated on INT8 validation scores
    thresholds=THRESHOLDS,
    window_size=WINDOW_SIZE,
    run_tag=RUN_TAG,
    test_symbols=y_test_symbols,
    deploy_threshold_name="precision_floor_90",
    rr_mean=rr_mean, rr_std=rr_std,
    n_calib=800,
    n_boot=2000,
    batch_one=True,   # mirrors how the firmware invokes the model, one window at a time
    out_dir=".",
)

## Optional: precision/sensitivity sweep over the loss and class-weight knobs

The model is over on sensitivity (0.965 against a 0.95 target) and under on precision
(0.852 against 0.90), so the settings biasing it toward recall are being paid for in the
wrong currency. This sweeps them.

Two knobs:

- `alpha` — the focal-loss weight on the positive term. Lower favours precision, higher
  favours recall. Currently 0.3.
- `pos_boost` — the manual multiplier on top of balanced class weights. Currently 1.3.
  **Setting it to 1.0 removes the recall bias entirely**, which is the cleanest precision
  lever available.

`pos_boost` only started doing anything once `focal_loss` was fixed to return one value per
sample — see its docstring. Before that fix every class-weight setting here was inert, so
any earlier tuning of these numbers told you nothing.

**The sweep never touches the test set.** Validation is split in two: one half chooses the
operating point, the other scores the configuration. Ranking on the same beats used to pick
the threshold would flatter every configuration and flatter the overfitted ones most.
Only `finalize_on_test` reads test data, once, after the winner is settled. If you re-run
the sweep after seeing test numbers, the test set has become a second validation set and
the write-up should say so.

Cost is one training run per (config, seed): the default 6-config grid at one seed is about
six runs. Seed noise here is large, so `n_seeds=2` ranks more reliably at double the cost.
Results checkpoint after every run; `resume=True` continues an interrupted session.

In [ ]:
RUN_SWEEP = False   # flip to True to run the sweep
SWEEP_SEEDS = 1

if RUN_SWEEP:
    from tibok.sweep import make_grid, run_sweep, summarize_sweep, print_sweep

    def sweep_train_one(cfg, seed):
        tf.keras.utils.set_random_seed(seed)
        m = build_model(WINDOW_SIZE)
        m.compile(
            optimizer=Adam(learning_rate=3e-4),
            loss=focal_loss(gamma=cfg["gamma"], alpha=cfg["alpha"]),
            metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                     tf.keras.metrics.Recall(name='recall')],
        )
        cw = compute_class_weight(class_weight='balanced',
                                  classes=np.unique(y_train_aug), y=y_train_aug)
        d = {i: w for i, w in enumerate(cw)}
        d[1] *= cfg["pos_boost"]
        m.fit([X_train_aug, RR_train_aug], y_train_aug,
              validation_data=([X_val, RR_val_n], y_val),
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6,
                                                          restore_best_weights=True),
                         tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                              patience=3, min_lr=1e-6)],
              class_weight=d, verbose=0)
        return m

    sweep_grid = make_grid(pos_boosts=(1.0, 1.3), alphas=(0.15, 0.30, 0.45))
    sweep_rows = run_sweep(sweep_train_one, sweep_grid, X_val, RR_val_n, y_val,
                           window_size=WINDOW_SIZE, n_seeds=SWEEP_SEEDS,
                           out_dir=".", run_tag=RUN_TAG, resume=True)
    sweep_summary = summarize_sweep(sweep_rows)
    print_sweep(sweep_summary)
else:
    sweep_summary = None
    print("RUN_SWEEP is False -- skipping the loss/class-weight sweep.")

## Optional: R independent trials, for a variance-aware answer to RQ2.1/RQ2.3

Everything above is **one** model, so it supports "quantization cost F1 0.040 *in this
run*" and nothing stronger. This section repeats the whole train -> quantize -> evaluate
experiment `N_TRIALS` times with different seeds and reports mean +/- SD, so the claim
becomes "F1 0.040 +/- <sd> across N runs".

**This is not the same as raising `N_CANDIDATES`.** That is a best-of-N *search*: it trains
N models, keeps the best on validation PR-AUC, and discards the rest. Raising it gives one
model chosen from a bigger pool, and makes the winner's validation PR-AUC *more*
optimistically biased, since you report the maximum of N noisy draws from the same set you
selected on. It adds no evidence about reproducibility. A trial is an independent
replication and does.

The patient split is held fixed across trials on purpose -- it is the split the methodology
commits to, and re-drawing it per trial would change the study population. So what is
measured here is **training variance** (initialization, augmentation draws, shuffling), not
variance across patient populations. State which one you are reporting.

Cost: roughly one training run per trial. At ~8 min/run, `N_TRIALS = 10` is about 80
minutes. Results are checkpointed to `<RUN_TAG>_trials.json` after every trial and
`resume=True` picks up where a disconnected runtime stopped, so this survives Colab
dropping the session partway.

In [ ]:
RUN_TRIALS = False   # flip to True to run the replication study
N_TRIALS = 10
CANDIDATES_PER_TRIAL = 1  # >1 keeps best-of-N inside each trial; cost multiplies

if RUN_TRIALS:
    from tibok.trials import run_trials, summarize_trials, print_trial_summary

    def trial_build(seed):
        tf.keras.utils.set_random_seed(seed)
        return build_model(WINDOW_SIZE)

    def trial_fit(model, seed):
        model.fit(
            [X_train_aug, RR_train_aug], y_train_aug,
            validation_data=([X_val, RR_val_n], y_val),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6,
                                                        restore_best_weights=True),
                       tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                            patience=3, min_lr=1e-6)],
            class_weight=class_weight_dict, verbose=0,
        )

    trial_rows = run_trials(
        trial_build, trial_fit,
        X_train_aug, RR_train_aug, y_train_aug,
        X_val, RR_val_n, y_val,
        X_test, RR_test_n, y_test,
        window_size=WINDOW_SIZE, n_trials=N_TRIALS,
        candidates_per_trial=CANDIDATES_PER_TRIAL,
        out_dir=".", run_tag=RUN_TAG, resume=True,
    )
    trial_summary = summarize_trials(trial_rows)
    print_trial_summary(trial_summary)
else:
    trial_summary = None
    print("RUN_TRIALS is False -- skipping the replication study.")

## Save + download everything

`quantize_and_test` has already written `<RUN_TAG>_model_int8.tflite`, `<RUN_TAG>_model_int8.h`
and `<RUN_TAG>_quantization_report.json`. The training summary below is merged with the
quantization report so one JSON carries the whole run.

In [ ]:
summary = {
    "candidate_val_pr_aucs": [float(v) for v in candidate_val_pr_aucs],
    "best_val_pr_auc": float(best_val_pr_auc),
    "thresholds": THRESHOLDS,
    "deploy_threshold_recommended": "precision_floor_90",
    "test_results": results,
    "symbol_breakdown": symbol_breakdown,
    "rr_feature_norm": {"mean": rr_mean.tolist(), "std": rr_std.tolist()},
    "quantization": quant_report,
    "trials": trial_summary,
    "sweep": sweep_summary,
}
with open(f'{RUN_TAG}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

from google.colab import files
for fname in [f'{RUN_TAG}_model_int8.h', f'{RUN_TAG}_model_int8.tflite',
              f'{RUN_TAG}_quantization_report.json', f'{RUN_TAG}_summary.json']:
    if os.path.exists(fname):
        files.download(fname)